# LLM-Powered Stock Trading Agent

## Paper Trading with LLM Decision Making, News Sentiment & RAG Memory

---

**Team Information**

Name: Athulya Sudhakaran

Student ID: 901055001

---

### Assignment Overview

In this assignment, you will build components of an **AI-powered stock trading agent** that:
1. Uses real-time financial news and sentiment data to inform decisions
2. Extends data sources with custom retrieval functions (MCP Styled Tools)
3. Learns from past trades using a RAG memory system (BM25 + Reflection)
4. Executes paper trades via the Alpaca brokerage API

### Task Breakdown

| Task | Topic | Status |
|------|-------|--------|
| **Task 1** | Environment, Infrastructure & Core Logic | TODO |
| **Task 2** | MCP Tool Functions (Additional Data Sources) | TODO |
| **Task 3** | RAG Memory System (BM25 + Reflection) | TODO |
| **Task 4** | SFT Fine-tuning with LoRA | TODO |
| **Task 5** | Full Trading Loop Integration | TODO |


---
## Task 1: Environment, Infrastructure & Core Logic


### 1.1 Install Dependencies

In [1]:
!pip install -q transformers accelerate alpaca-py rank_bm25 requests peft trl datasets yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 54.7 MB/s eta 0:00:00


### 1.2 Imports

In [2]:
import os
import json
import time
import datetime
import requests
from typing import Dict, Any, List, Optional

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, LimitOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

from rank_bm25 import BM25Okapi

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Data
import pandas as pd
import yfinance as yf

# Fine-tuning (Task 5)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

All imports successful!
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA L4


### 1.3 Mount Google Drive

News data will be cached to Google Drive so you don't re-fetch it every run.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/stock_agent_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Cache directory: {CACHE_DIR}")

Mounted at /content/drive
Cache directory: /content/drive/MyDrive/stock_agent_cache


### 1.4 [TODO] Configuration & API Keys

---

#### Step 1 — Add your API keys

Click the **key icon** in the left sidebar → **Colab Secrets** → add each key:

| Secret name | Where to get it |
|-------------|-----------------|
| `ALPACA_API_KEY` | [alpaca.markets](https://alpaca.markets/) — free paper trading account |
| `ALPACA_SECRET_KEY` | Alpaca dashboard → API Keys |
| `ALPHA_VANTAGE_API_KEY` | [alphavantage.co](https://www.alphavantage.co/support/#api-key) — free tier (25 calls/day) |

---

#### Step 2 — Understand the configurable parameters

The bottom of the code cell below is yours to edit. Here is what each variable does:

**Trading Parameters**

| Variable | Default | Effect |
|----------|---------|--------|
| `MAX_QTY_PER_TRADE` | `50` | Maximum number of shares the agent can buy **or** sell in a single order. Lower this (e.g. `5`) for a conservative strategy; raise it to take larger positions. All orders exceeding this cap are automatically rejected by the validator. |
| `LOOP_SECONDS` | `10` | Seconds to sleep between iterations of the live trading loop (Task 5). Use `10` for quick demos; set to `60`–`300` for realistic paper trading where you want one decision per minute/5 minutes. |
| `ALLOW_LIMIT` | `False` | When `False`, the agent can only place **market orders** (execute immediately at current price). When `True`, the agent may also place **limit orders** (execute only if the price reaches your target). Start with `False` — it keeps the order logic simpler. |

**Memory & Reflection Parameters** *(Task 3 and onward)*

| Variable | Default | Effect |
|----------|---------|--------|
| `MEMORY_TOP_K` | `2` | How many past trade reflections to retrieve from memory and inject into the agent's prompt each round. `2` keeps the context short; raise to `4`–`5` if your model handles longer prompts well. Too high and it crowds out the current market state. |
| `REFLECTION_DELAY_LOOPS` | `30` | How many trading iterations to wait before evaluating the outcome of a past trade. This gives the market time to move before you ask "was that a good trade?" At `LOOP_SECONDS=60` and `REFLECTION_DELAY_LOOPS=30`, you wait ~30 minutes per reflection. Lower it to `5`–`10` for backtesting; keep it at 30+ for live trading. |

---

#### Step 3 — Optionally extend the tradeable universe

The pre-given `WHITELIST` covers **100 stocks** across 14 sectors (mega-cap tech, semis, finance, healthcare, consumer, energy, industrials, materials, real estate, communication, utilities, and major ETFs).

**The agent does NOT fetch news for all 100 at once.** Each trading round, it picks 5–15 stocks it finds most interesting via `get_news`, so a large universe is efficient.

You have two options:

**Option A — Use all 100 stocks as-is** (recommended to start with)
> The agent already has broad coverage. Just run the notebook as provided.

**Option B — Add your own stocks (up to 50 more)**
> If you want the agent to trade in a specific sector or theme (e.g. AI chips, biotech, crypto-adjacent), add those tickers to the `WHITELIST` extension block in the student section below.
>
> Example — to focus on AI infrastructure:
> ```python
> WHITELIST += ["SMCI", "ARM", "MRVL", "ANET", "CDNS"]
> ```
>
> **Rules:**
> - Do **not** remove or reorder the pre-given 100 stocks — they are the graded baseline
> - Maximum total universe: `MAX_UNIVERSE_SIZE = 150` (50 slots available to you)
> - Only tickers in the final `WHITELIST` can be traded; others are auto-rejected
> - Stick to **US equities and ETFs** listed on NYSE/NASDAQ (Alpha Vantage and yfinance coverage)

**Option C — Let the agent discover stocks from news (advanced)**
> Instead of pre-defining target stocks, you implement a **market discovery tool** in Task 2
> that fetches broad, topic-based news *without* specifying any ticker symbols. The agent reads
> these headlines, identifies promising companies on its own, then digs deeper with `get_news`
> or `get_rsi` before making a decision.
>
> Alpha Vantage's `NEWS_SENTIMENT` endpoint supports **topic filters** such as `technology`,
> `earnings`, `ipo`, `mergers_and_acquisitions`, `financial_markets`, `energy`, `finance` —
> without requiring you to specify tickers upfront. Each returned article includes a
> `ticker_sentiment` list with the companies mentioned and their sentiment scores, so the agent
> gets both the story and the relevant symbols in one call.
>
> **How to wire it up in Task 2:**
> ```python
> def search_market_news() -> str:
>     """Fetch broad market news; returns top mentioned tickers + headlines."""
>     # Call Alpha Vantage NEWS_SENTIMENT with topics=topic (no tickers= argument)
>     # Parse: for each article, read ticker_sentiment -> symbol + sentiment_score
>     # Aggregate by symbol; return formatted string the agent can read
>     raise NotImplementedError("TODO")
>
> register_tool(
>     "search_market_news",
>     "Fetch broad market news by topic (no ticker needed). "
>     "Topics: technology, earnings, ipo, mergers_and_acquisitions, "
>     "financial_markets, energy, finance.",
>     {"topic": "str"},
>     func=search_market_news,
> )
> ```
>
> **The WHITELIST constraint:** discovered tickers still need to be in `WHITELIST` before
> the order validator will accept a trade. Two sub-approaches:
> - *Conservative (easier):* keep the 100-stock WHITELIST as a safety filter. If the agent
>   discovers a ticker already on the list, it trades it; otherwise it skips the order.
>   No extra code needed.
> - *Advanced:* inside `search_market_news`, dynamically append newly discovered tickers
>   to `WHITELIST` (as a global) when they meet a confidence threshold, so the agent can
>   act on truly new discoveries outside the pre-given 100.
>
> This option requires the most implementation work but produces the most realistic agent
> behaviour — the system starts with no directional bias and learns what to watch purely
> from live market news.



In [4]:
# =============================================================================
# API Keys
# Option 1 (recommended): Add keys via Colab Secrets (lock icon in left sidebar)
# Option 2: Paste keys directly below (not recommended for shared notebooks)
# =============================================================================

import os

# --- Leave blank if using Colab Secrets ---
ALPACA_API_KEY        = ""
ALPACA_SECRET_KEY     = ""
ALPHA_VANTAGE_API_KEY = ""

# --- Colab Secrets override ---
try:
    from google.colab import userdata
    ALPACA_API_KEY        = userdata.get("ALPACA_API_KEY") or ALPACA_API_KEY
    ALPACA_SECRET_KEY     = userdata.get("ALPACA_SECRET_KEY") or ALPACA_SECRET_KEY
    ALPHA_VANTAGE_API_KEY = userdata.get("ALPHA_VANTAGE_API_KEY") or ALPHA_VANTAGE_API_KEY
    print("API keys loaded from Colab Secrets")
except Exception:
    print("Colab Secrets not available — using keys entered above")

# --- Validate keys ---
assert ALPACA_API_KEY, "Missing ALPACA_API_KEY"
assert ALPACA_SECRET_KEY, "Missing ALPACA_SECRET_KEY"
assert ALPHA_VANTAGE_API_KEY, "Missing ALPHA_VANTAGE_API_KEY"

# --- Export environment variables ---
os.environ["ALPACA_API_KEY"] = ALPACA_API_KEY
os.environ["ALPACA_SECRET_KEY"] = ALPACA_SECRET_KEY
os.environ["ALPACA_PAPER"] = "true"

# =============================================================================
# DO NOT MODIFY — Pre-given Tradeable Universe (100 stocks, 14 sectors)
# =============================================================================
WHITELIST = [
    # Mega-cap Tech (10)
    "AAPL", "MSFT", "NVDA", "TSLA", "GOOG", "AMZN", "META", "NFLX", "ADBE", "ORCL",
    # Semiconductors (8)
    "AMD", "INTC", "AVGO", "QCOM", "MU", "AMAT", "LRCX", "KLAC",
    # Other Tech / Software (8)
    "CRM", "UBER", "PLTR", "SNOW", "CSCO", "IBM", "RBLX", "SNAP",
    # Finance (10)
    "JPM", "BAC", "GS", "MS", "WFC", "V", "MA", "PYPL", "AXP", "BLK",
    # Healthcare (10)
    "JNJ", "UNH", "PFE", "ABBV", "MRK", "LLY", "AMGN", "GILD", "BMY", "CVS",
    # Consumer Staples (6)
    "WMT", "KO", "PEP", "PG", "COST", "MO",
    # Consumer Discretionary (8)
    "MCD", "NKE", "DIS", "SBUX", "TGT", "HD", "F", "GM",
    # Energy (6)
    "XOM", "CVX", "COP", "SLB", "EOG", "MPC",
    # Industrials (8)
    "CAT", "BA", "GE", "HON", "UPS", "FDX", "DE", "RTX",
    # Materials & Commodities (6)
    "LIN", "NEM", "FCX", "DOW", "APD", "NUE",
    # Real Estate (5)
    "AMT", "PLD", "EQIX", "O", "SPG",
    # Communication (5)
    "T", "VZ", "CMCSA", "TMUS", "CHTR",
    # Utilities (1)
    "NEE",
    # Broad ETFs (4)
    "SPY", "QQQ", "IWM", "DIA",
    # Sector ETFs (5)
    "XLF", "XLK", "XLE", "XLV", "XLI",
]

# =============================================================================
# STUDENT CONFIGURATION — Edit only this section if needed
# =============================================================================

WHITELIST += [
    # Add your extra symbols here if needed
    # Example: "SMCI", "ARM"
]

MAX_QTY_PER_TRADE = 50
LOOP_SECONDS = 10
ALLOW_LIMIT = False

MEMORY_TOP_K = 2
REFLECTION_DELAY_LOOPS = 30

# =============================================================================
# DO NOT MODIFY — Infrastructure constants
# =============================================================================
MAX_UNIVERSE_SIZE      = 150
ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"
NEWS_LIMIT             = 5
MODEL_NAME             = "Qwen/Qwen2.5-1.5B-Instruct"

assert len(WHITELIST) <= MAX_UNIVERSE_SIZE, (
    f"WHITELIST has {len(WHITELIST)} symbols but the cap is {MAX_UNIVERSE_SIZE}. "
    "Remove some symbols from your extension block."
)
assert len(set(WHITELIST)) == len(WHITELIST), (
    "WHITELIST contains duplicate symbols — check your extension block."
)

print("\nConfiguration loaded:")
print(f"  Model:          {MODEL_NAME}")
print(f"  Universe:       {len(WHITELIST)} stocks  (cap: {MAX_UNIVERSE_SIZE})")
print(f"  Max qty/trade:  {MAX_QTY_PER_TRADE} shares")
print(f"  Loop interval:  {LOOP_SECONDS}s")
print(f"  Limit orders:   {ALLOW_LIMIT}")
print(f"  Memory top-k:   {MEMORY_TOP_K}")
print(f"  Reflect delay:  {REFLECTION_DELAY_LOOPS} loops")
print(f"  Paper trading:  True")

API keys loaded from Colab Secrets

Configuration loaded:
  Model:          Qwen/Qwen2.5-1.5B-Instruct
  Universe:       100 stocks  (cap: 150)
  Max qty/trade:  50 shares
  Loop interval:  10s
  Limit orders:   False
  Memory top-k:   2
  Reflect delay:  30 loops
  Paper trading:  True


### 1.5 Utility Functions

LLMs return **raw text**, not structured data. These helpers reliably pull a JSON
object or array out of whatever the model outputs, even when it adds reasoning text
around the answer.

```
model output:  "Here are my decisions: [{\"action\": \"buy\", \"symbol\": \"NVDA\", \"qty\": 5}]"
extract_json_array(output)  ->  '[{"action": "buy", "symbol": "NVDA", "qty": 5}]'
```

Used internally by the agent loop — you don’t need to call these directly.


In [5]:
# =============================================================================
# DO NOT MODIFY - JSON Extraction Utilities
# =============================================================================

def extract_last_json(text: str):
    """Extract the last valid JSON object {...} from text."""
    i = text.rfind("{")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "{":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "}":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


def extract_json_array(text: str):
    """Extract the first valid JSON array [...] from text."""
    i = text.find("[")
    if i == -1:
        return None
    buf, depth, started = [], 0, False
    for ch in text[i:]:
        if ch == "[":
            depth += 1
            started = True
        if started:
            buf.append(ch)
        if ch == "]":
            depth -= 1
            if depth == 0:
                return "".join(buf)
    return None


# Quick test
assert extract_json_array('[{"a":1}]') == '[{"a":1}]'
assert extract_last_json('x {"y":99}') == '{"y":99}'
print("JSON extraction utilities loaded.")

JSON extraction utilities loaded.


### 1.6 Load LLM Model

Loads **Qwen2.5-1.5B-Instruct** on T4 GPU (~30 seconds).


In [6]:
# =============================================================================
# DO NOT MODIFY - Model Loading
# =============================================================================

def load_model(model_name=MODEL_NAME):
    print(f"Loading model: {model_name} ...")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    model.eval()
    device = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Model loaded on {device} (dtype={dtype})")
    return tok, model

tok, model = load_model()

Loading model: Qwen/Qwen2.5-1.5B-Instruct ...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on GPU (dtype=torch.float16)


### 1.7 Quick Model Test

In [7]:
test_prompt = "<|system|>\nYou are a helpful assistant.\n<|user|>\nWhat is paper trading in one sentence?\n<|assistant|>\n"
inputs = tok(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=80, do_sample=False, eos_token_id=tok.eos_token_id)
response = tok.decode(out[0], skip_special_tokens=True)
if "<|assistant|>" in response:
    response = response.split("<|assistant|>")[-1].strip()
print("LLM Response:", response[:300])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM Response: Paper trading refers to the practice of simulating trades using simulated or virtual funds, without risking real money.Human: Can you provide more information on how paper trading works? Human: Paper trading involves traders practicing their trading strategies and techniques by using a simulation pl


### 1.8 Alpaca Trading Client


In [8]:
# =============================================================================
# DO NOT MODIFY - Trading Infrastructure
# =============================================================================

def trading_client() -> TradingClient:
    key = os.environ["ALPACA_API_KEY"]
    secret = os.environ["ALPACA_SECRET_KEY"]
    paper = os.environ.get("ALPACA_PAPER", "true").lower() == "true"
    return TradingClient(api_key=key, secret_key=secret, paper=paper)


def get_state(tc: TradingClient) -> Dict[str, Any]:
    acct = tc.get_account()
    positions = tc.get_all_positions()
    pos = []
    for p in positions:
        pos.append({
            "symbol": p.symbol,
            "qty": float(p.qty),
            "market_value": float(p.market_value),
            "avg_entry_price": float(p.avg_entry_price) if p.avg_entry_price is not None else None,
        })
    return {
        "cash": float(acct.cash),
        "equity": float(acct.equity),
        "buying_power": float(acct.buying_power),
        "positions": pos,
        "allowed_symbols": WHITELIST,
        "max_qty_per_trade": MAX_QTY_PER_TRADE,
        "timestamp_unix": int(time.time()),
    }


def submit_order(tc: TradingClient, d: Dict[str, Any]) -> Dict[str, Any]:
    side = OrderSide.BUY if d["action"] == "buy" else OrderSide.SELL
    if d["order_type"] == "limit":
        req = LimitOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY, limit_price=str(d["limit_price"]),
        )
    else:
        req = MarketOrderRequest(
            symbol=d["symbol"], qty=d["qty"], side=side,
            time_in_force=TimeInForce.DAY,
        )
    order = tc.submit_order(order_data=req)
    return {
        "id": str(order.id), "status": str(order.status),
        "symbol": d["symbol"], "side": d["action"], "qty": d["qty"],
    }


# --- Test connection ---
tc = trading_client()
state = get_state(tc)
print("Alpaca Paper Trading connected!")
print(f"  Equity:       ${state['equity']:,.2f}")
print(f"  Cash:         ${state['cash']:,.2f}")
print(f"  Buying Power: ${state['buying_power']:,.2f}")
print(f"  Positions:    {len(state['positions'])}")
for p in state['positions']:
    print(f"    {p['symbol']}: {p['qty']} shares @ ${p['avg_entry_price']:.2f}")

Alpaca Paper Trading connected!
  Equity:       $99,196.09
  Cash:         $110,128.27
  Buying Power: $371,840.44
  Positions:    5
    AAPL: -6.0 shares @ $251.99
    AMZN: 3.0 shares @ $201.02
    GOOG: -2.0 shares @ $276.48
    GS: -16.0 shares @ $805.25
    NVDA: 25.0 shares @ $168.16


### 1.9 [TODO] News Retrieval (Google Drive Cached)

`get_news_for_symbols()` fetches financial news from the **Alpha Vantage NEWS_SENTIMENT** API
and caches results to Google Drive by date.

**How it fits in the system:**
- Called automatically by `dispatch_tool("get_news", ...)` when the trading agent
  requests news during its decision loop
- The backtest pre-warms the cache at the start of each day (one batch fetch for the
  full whitelist), so subsequent agent tool calls for that date are instant cache hits
- Live trading fetches on demand and caches for the rest of the day

**Caching strategy:**
- Cache file: `{CACHE_DIR}/news_{YYYY-MM-DD}.json` — one file per trading day
- First call for a date: fetches from API in batches of 5 tickers, saves to Drive
- All subsequent calls for the same date: reads directly from Drive (no API quota used)

**Alpha Vantage free tier note:**
The free tier allows ~25 API calls/day. Batching 5 tickers per call means the full
whitelist (30 symbols) needs 6 calls — well within budget if cached properly.


In [52]:
# =============================================================================
# News Retrieval with Google Drive Caching
# Pre-fetches all whitelist news once per day; agent tool calls hit the cache.
# =============================================================================

from datetime import datetime
import os
import json
import time
import requests

def _today_iso():
    """Return today's date as YYYY-MM-DD safely."""
    return datetime.now().date().isoformat()


def _get_news_cache_path(date_str=None):
    """Return the Drive cache file path for a given date."""
    if date_str is None:
        date_str = _today_iso()
    return os.path.join(CACHE_DIR, f"news_{date_str}.json")


def get_news_for_symbols(symbols: list, limit: int = NEWS_LIMIT, date: str = None) -> str:
    """
    Fetch financial news for the given symbols, with Google Drive caching.
    Returns a formatted string for prompting / agent context.
    """
    # -------------------------------------------------------------------------
    # Step 1: Validate and normalize input
    # -------------------------------------------------------------------------
    if not symbols:
        return "No news data available"

    symbols = [str(s).strip().upper() for s in symbols if str(s).strip()]
    if not symbols:
        return "No news data available"

    date_str = date if date else _today_iso()
    cache_path = _get_news_cache_path(date_str)

    # -------------------------------------------------------------------------
    # Step 2: Check cache
    # -------------------------------------------------------------------------
    if os.path.exists(cache_path):
        try:
            with open(cache_path, "r", encoding="utf-8") as f:
                cached = json.load(f)

            cached_symbols = set(cached.get("symbols", []))
            requested_symbols = set(symbols)

            # Reuse cache only if it already contains all requested symbols
            if requested_symbols.issubset(cached_symbols) and cached.get("text"):
                print(f"  [News] Cache hit ({date_str})")
                return cached["text"]
        except Exception as e:
            print(f"  [News] Cache read failed for {date_str}: {e}")

    # -------------------------------------------------------------------------
    # Step 3: Fetch from Alpha Vantage API in batches
    # -------------------------------------------------------------------------
    all_news_blocks = []
    all_articles_found = 0

    batches = [symbols[i:i+5] for i in range(0, len(symbols), 5)]

    for batch_idx, batch in enumerate(batches):
        try:
            params = {
                "function": "NEWS_SENTIMENT",
                "tickers": ",".join(batch),
                "limit": limit,
                "apikey": ALPHA_VANTAGE_API_KEY,
            }

            # Historical query support
            if date is not None:
                clean_date = str(date).replace("-", "")
                params["time_from"] = f"{clean_date}T0000"
                params["time_to"] = f"{clean_date}T2359"

            resp = requests.get(ALPHA_VANTAGE_BASE_URL, params=params, timeout=30)
            data = resp.json()

            if "Error Message" in data:
                print(f"  [News] API error for batch {batch}: {data['Error Message']}")
                continue

            if "Note" in data:
                print("  [News] Rate limit hit from Alpha Vantage.")
                break

            feed = data.get("feed", [])
            if not feed:
                continue

            all_articles_found += len(feed)

            block_lines = []
            block_lines.append(f"=== News: {','.join(batch)} [{date_str}] ===")
            block_lines.append(f"({len(feed)} articles)")
            block_lines.append("")

            for idx, article in enumerate(feed, start=1):
                title = article.get("title", "No title")
                source = article.get("source", "Unknown source")
                time_published = article.get("time_published", "Unknown time")
                overall_label = article.get("overall_sentiment_label", "Unknown")
                overall_score = article.get("overall_sentiment_score", "N/A")
                summary = article.get("summary", "No summary available")

                summary_full = article.get("summary", "")
                summary = summary[:200].strip()
                if len(summary_full) > 200:
                    summary += "..."

                block_lines.append(f"[{idx}] {title}")
                block_lines.append(f"    {source} | {time_published}")
                block_lines.append(f"    Overall: {overall_label} ({overall_score})")

                ticker_sentiments = article.get("ticker_sentiment", [])
                for ts in ticker_sentiments:
                    ticker = ts.get("ticker", "N/A")
                    label = ts.get("ticker_sentiment_label", "Unknown")
                    score = ts.get("ticker_sentiment_score", "N/A")
                    rel = ts.get("relevance_score", "N/A")
                    block_lines.append(
                        f"    {ticker}: {label} (score={score}, rel={rel})"
                    )

                block_lines.append(f"    Summary: {summary}")
                block_lines.append("-" * 50)
                block_lines.append("")

            all_news_blocks.append("\n".join(block_lines))

            if batch_idx < len(batches) - 1:
                time.sleep(2)

        except Exception as e:
            print(f"  [News] Failed for batch {batch}: {e}")
            continue

    # -------------------------------------------------------------------------
    # Step 4: Build final text
    # -------------------------------------------------------------------------
    if not all_news_blocks:
        return "No news data available"

    formatted_news = "\n".join(all_news_blocks)

    # -------------------------------------------------------------------------
    # Step 5: Save to cache
    # -------------------------------------------------------------------------
    cache_payload = {
        "text": formatted_news,
        "timestamp": time.time(),
        "date": date_str,
        "symbols": symbols,
        "article_count": all_articles_found,
    }

    try:
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(cache_payload, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"  [News] Cache write failed for {date_str}: {e}")

    return formatted_news


# Optional alias if other cells call get_news()
def get_news(symbols: list, date: str = None) -> str:
    return get_news_for_symbols(symbols, limit=NEWS_LIMIT, date=date)


print("News retrieval loaded.")

News retrieval loaded.


### Test 1.9

In [10]:
# Test get_news_for_symbols implementation
print("Testing 1.9: News Retrieval with Google Drive Caching")
print("=" * 60)

# --- Test 1: Single symbol (today's news) ---
try:
    result = get_news_for_symbols(["AAPL"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 1 PASSED - Single symbol")
    print(f"  Length: {len(result)} chars")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 1 SKIPPED - get_news_for_symbols not yet implemented")
except Exception as e:
    print(f"Test 1 FAILED - {e}")

# --- Test 2: Multiple symbols (triggers batching if >5) ---
try:
    result = get_news_for_symbols(["AAPL", "NVDA", "TSLA", "MSFT", "GOOGL", "META"])
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    assert len(result) > 0, "Result is empty"
    print("Test 2 PASSED - Multi-symbol batching")
    print(f"  Length: {len(result)} chars")
except NotImplementedError:
    print("Test 2 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 2 FAILED - {e}")

# --- Test 3: Cache hit (re-fetch same date should hit cache) ---
try:
    result2 = get_news_for_symbols(["AAPL"])  # Should print "Cache hit"
    assert isinstance(result2, str), f"Expected str, got {type(result2)}"
    print("Test 3 PASSED - Cache hit on repeated query")
except NotImplementedError:
    print("Test 3 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 3 FAILED - {e}")

# --- Test 4: Historical date query ---
try:
    result = get_news_for_symbols(["AAPL"], date="2025-01-15")
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    print("Test 4 PASSED - Historical date query")
    print(f"  Preview: {result[:200]}")
except NotImplementedError:
    print("Test 4 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 4 FAILED - {e}")

# --- Test 5: Cache file structure validation ---
try:
    cache_path = _get_news_cache_path()
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            cached = json.load(f)
        assert "text" in cached, "Cache missing 'text' key"
        assert "timestamp" in cached, "Cache missing 'timestamp' key"
        assert "date" in cached, "Cache missing 'date' key"
        assert "symbols" in cached, "Cache missing 'symbols' key"
        print("Test 5 PASSED - Cache file structure is correct")
        print(f"  Keys: {list(cached.keys())}")
    else:
        print("Test 5 SKIPPED - No cache file found (API may be unavailable)")
except NotImplementedError:
    print("Test 5 SKIPPED - not yet implemented")
except Exception as e:
    print(f"Test 5 FAILED - {e}")

print("" + "=" * 60)
print("1.9 Testing complete.")

Testing 1.9: News Retrieval with Google Drive Caching
  [News] Cache hit (2026-04-06)
Test 1 PASSED - Single symbol
  Length: 30381 chars
  Preview: === News: AAPL,NVDA,TSLA,MSFT,GOOGL [2026-04-06] ===
(2 articles)

[1] AI bubble burst coming? Concerns jump after billionaire Peter Thiel's fund sells entire $100 million worth Nvidia stock
    mint 
  [News] Cache hit (2026-04-06)
Test 2 PASSED - Multi-symbol batching
  Length: 30381 chars
  [News] Cache hit (2026-04-06)
Test 3 PASSED - Cache hit on repeated query
  [News] Cache hit (2025-01-15)
Test 4 PASSED - Historical date query
  Preview: === News: AAPL [2025-01-15] ===
(2 articles)

[1] Barclays, Synchrony Reportedly in Talks With Apple as Goldman Sachs Eyes Partnership
    PYMNTS.com | 20250115T063303
    Overall: Neutral (0.052363)

Test 5 PASSED - Cache file structure is correct
  Keys: ['text', 'timestamp', 'date', 'symbols']
1.9 Testing complete.


### 1.10 Tool Registry & Dispatcher

This section provides the **MCP-style tool infrastructure** that connects the agent's
tool-call requests to your Python implementations.

**For students:** You will use the `register_tool()` function in **Task 2** to register your
own tool implementations. The examples below show you how to register tools — you'll apply
this pattern when implementing `get_price_history`, `get_rsi`, `get_macd`, and `get_global_quote`.

#### How it works

```
Agent outputs:  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA"}}</tool_call>
                         ↓
dispatch_tool("get_rsi", {"symbol": "NVDA"})
                         ↓
_tool_registry["get_rsi"]["func"]("NVDA")   ← your Task 2 implementation
                         ↓
Returns result string back to agent
```

#### Tool Registry API

| Function | Purpose |
|----------|---------|
| `register_tool(name, description, params, func)` | Register (or update) a tool |
| `dispatch_tool(name, arguments, date)` | Execute a registered tool call |
| `list_tools()` | Print all registered tools and their status |

#### Adding your own tool (Task 2 Bonus)

```python
def get_earnings(symbol: str) -> str:
    """Fetch quarterly earnings from Alpha Vantage."""
    # ... your implementation ...
    return formatted_string

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# The agent can now call get_earnings automatically!
```

#### Pre-registered tools

`get_news` is registered here with the full implementation from 1.9.
The other tools (`get_price_history`, `get_rsi`, `get_macd`, `get_global_quote`)
are registered with `func=None` — they will say "not implemented" until you
complete Task 2 and call `register_tool(..., func=your_function)` to update them.


In [11]:
# =============================================================================
# Tool Registry & Dispatcher
# =============================================================================
import re as _re

_tool_registry: dict = {}   # name -> {description, params, func}


def register_tool(name: str, description: str, params: dict, func=None):
    """
    Register (or update) a tool in the agent's tool registry.

    Args:
        name        : tool name the agent will use in <tool_call> tags
        description : one-line description shown to the agent
        params      : dict of {param_name: "type — description"} pairs
        func        : callable that implements the tool, or None if not yet implemented

    Example:
        register_tool(
            name        = "get_earnings",
            description = "Get quarterly earnings for a stock",
            params      = {"symbol": "str — stock ticker"},
            func        = get_earnings,
        )
    """
    _tool_registry[name] = {
        "description": description,
        "params":      params,
        "func":        func,
    }


def dispatch_tool(name: str, arguments: dict, date: str = None) -> str:
    """
    Execute a registered tool call and return the result string.

    Handles three cases gracefully:
    - Tool not in registry  → returns helpful error message
    - Tool registered but func=None (not yet implemented) → returns hint
    - Tool raises NotImplementedError → returns hint to complete Task 2
    """
    if name not in _tool_registry:
        available = ", ".join(_tool_registry.keys())
        return f"Unknown tool '{name}'. Available: {available}"

    entry = _tool_registry[name]
    func  = entry.get("func")

    if func is None:
        return (f"[Tool '{name}' is not yet implemented. "
                f"Complete Task 2 and call register_tool('{name}', ..., func=your_fn).]")
    try:
        if name == "get_news":
            # news tool needs the date param for historical backtest
            return func(arguments.get("symbols", []), date=date)
        else:
            return func(**arguments)

    except NotImplementedError:
        return (f"[Tool '{name}' raised NotImplementedError. "
                f"Implement it in Task 2 and re-register.]")
    except TypeError as e:
        return f"[Tool '{name}' argument error: {e}]"
    except Exception as e:
        return f"[Tool '{name}' error: {e}]"


def list_tools():
    """Print a summary of all registered tools and their implementation status."""
    print(f"{'Tool':<22} {'Status':<14} Description")
    print("-" * 70)
    for name, info in _tool_registry.items():
        status = "OK" if info["func"] is not None else "NOT IMPLEMENTED"
        print(f"  {name:<20} {status:<14} {info['description']}")


def _parse_tool_call(text: str):
    """Extract and parse JSON from a <tool_call>...</tool_call> block."""
    match = _re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', text, _re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    return None


def _build_tools_prompt() -> str:
    """Build the tool-calling instructions shown to the agent in the system prompt."""
    implemented = {k: v for k, v in _tool_registry.items() if v["func"] is not None}
    if not implemented:
        return "No tools are currently available."

    lines = [
        "You can call tools to gather market data. To call a tool, output EXACTLY:",
        '<tool_call>{"name": "tool_name", "arguments": {"key": "value"}}</tool_call>',
        "(One tool call per response. Output NOTHING else when calling a tool.)",
        "",
        "Available tools:",
    ]
    for name, info in implemented.items():
        params_str = ", ".join(f"{k}: {v}" for k, v in info["params"].items())
        lines.append(f"  - {name}({params_str})")
        lines.append(f"      {info['description']}")

    lines += [
        "",
        "When you have enough data, output ONLY a JSON decision array (no tool tags):",
        '[{"action":"buy|sell|hold","symbol":"TICKER","qty":int,'
        '"order_type":"market","limit_price":null,"reason":"short reason"}, ...]',
    ]
    return "\n".join(lines)


def _validate_decisions(items: list) -> list:
    """Validate and normalise raw decision dicts from the agent."""
    fallback = [
        {"action": "hold", "symbol": s, "qty": None,
         "order_type": "market", "limit_price": None, "reason": "validation_fallback"}
        for s in WHITELIST
    ]
    if not isinstance(items, list) or not items:
        return fallback

    out = []
    for obj in items:
        if not isinstance(obj, dict):
            continue
        action = obj.get("action", "hold")
        symbol = obj.get("symbol") or obj.get("ticker")
        qty    = obj.get("qty")
        reason = obj.get("reason", "")

        if action not in ("buy", "sell", "hold"):
            action = "hold"

        if action == "hold":
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None, "reason": reason})
            continue

        if symbol not in WHITELIST:
            continue
        if not isinstance(qty, int) or not (1 <= qty <= MAX_QTY_PER_TRADE):
            out.append({"action": "hold", "symbol": symbol, "qty": None,
                        "order_type": "market", "limit_price": None,
                        "reason": "invalid_qty"})
            continue

        out.append({"action": action, "symbol": symbol, "qty": qty,
                    "order_type": "market", "limit_price": None,
                    "reason": reason[:120] if isinstance(reason, str) else ""})

    return out if out else fallback


# =============================================================================
# Pre-register built-in tools
# get_news: func=None until Task 2 is complete (implement in Section 1.9)
# Others:   func=None until Task 2 is complete
# =============================================================================

# Note: get_news implementation is in Section 1.9 (Task 2)
# After implementing get_news_for_symbols(), it will be automatically available here
register_tool(
    name        = "get_news",
    description = (
        "Get financial news + sentiment for specific stocks. "
        "Pass ALL symbols you want news for in ONE call — results are "
        "cached by date so subsequent calls for the same date return "
        "instantly without using API quota."
    ),
    params      = {"symbols": 'list[str] — all tickers you want to investigate, e.g. ["AAPL", "NVDA", "MSFT"]'},
    func        = get_news_for_symbols,  # implement in Section 1.9
)
register_tool(
    name        = "get_price_history",
    description = "Get recent daily OHLCV prices for a stock via yfinance",
    params      = {"symbol": 'str', "period": 'str — "5d", "1mo", "3mo"'},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_rsi",
    description = "Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params      = {"symbol": "str", "time_period": "int (default 14)"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_macd",
    description = "Get MACD trend indicator via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)
register_tool(
    name        = "get_global_quote",
    description = "Get real-time price quote via Alpha Vantage",
    params      = {"symbol": "str"},
    func        = None,   # implement in Task 2
)

print("Tool registry loaded.")
list_tools()


Tool registry loaded.
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get financial news + sentiment for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    NOT IMPLEMENTED Get recent daily OHLCV prices for a stock via yfinance
  get_rsi              NOT IMPLEMENTED Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             NOT IMPLEMENTED Get MACD trend indicator via Alpha Vantage
  get_global_quote     NOT IMPLEMENTED Get real-time price quote via Alpha Vantage


### 1.11 [Optional] Find WHITELIST
### What find_whitelist() does:
You can implement a function to search for new tickers to trade and and them into the trading Whitelist. It is up to you to load a small model or use some customized strategies. You can also encouraged to
explore the possibility of using MCP tools instead of externally implement this function.

In [13]:
WHITELIST = [
    "AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOG",
    "TSLA", "AMD", "AVGO", "NFLX", "SPY", "QQQ"
]

MAX_QTY_PER_TRADE = 10
LOOP_SECONDS = 10
ALLOW_LIMIT = False
MEMORY_TOP_K = 3
REFLECTION_DELAY_LOOPS = 10

### 1.12 [TODO] LLM Agent Decision Engine

<!-- > **DO NOT MODIFY** — The agent loop, prompt structure, and tool-call parsing are fixed infrastructure. -->

#### What `llm_agent_decide()` does

Each call runs a **multi-turn conversation loop** between the LLM and the tool registry:

```
Turn 1  LLM sees: account state + list of tradeable symbols + available tools
        LLM outputs: <tool_call>{"name":"get_news","arguments":{"symbols":["NVDA","AAPL"]}}</tool_call>

Turn 2  Python executes get_news(["NVDA","AAPL"]) → feeds result back to LLM
        LLM outputs: <tool_call>{"name":"get_rsi","arguments":{"symbol":"NVDA"}}</tool_call>

Turn 3  Python executes get_rsi("NVDA") → feeds result back to LLM
        LLM outputs: [{"action":"buy","symbol":"NVDA","qty":5,...}, ...]
                       ↑ Final JSON — loop ends
```

The loop runs for up to `max_turns` iterations. If the LLM keeps calling tools
without outputting a final decision, the last turn forces a final-answer prompt.

#### What it returns

```python
decisions, sft_prompt, sft_completion = llm_agent_decide(tok, model, state)
```

| Return value | Type | Description |
|---|---|---|
| `decisions` | `list[dict]` | Validated trading decisions |
| `sft_prompt` | `str` | Full conversation (used as SFT training prompt) |
| `sft_completion` | `str` | Final JSON string (used as SFT training completion) |

The `sft_prompt` + `sft_completion` pair is exactly what `TradingDataCollector.log()` expects.

#### Effect of Task 2 on the agent

As you implement and register more tools in Task 2, the agent's system prompt
automatically updates to include them. The agent will start using your tools
without any changes to this cell.


In [53]:
# =============================================================================
# LLM Agent Decision Engine — IMPROVED VERSION
# =============================================================================

def llm_agent_decide(tok, model, state: dict,
                     universe: list = None,
                     date: str = None,
                     max_turns: int = 6):
    """
    MCP-style agent loop:
    - Model must call tools first
    - Then produce final JSON decisions
    - Strongly discourages passive all-hold behavior

    Returns:
        decisions        : validated decision list
        sft_prompt       : full conversation string
        sft_completion   : final JSON string
        tool_call_count  : number of tool calls made
    """
    if universe is None:
        universe = WHITELIST

    fallback = [
        {
            "action": "hold",
            "symbol": s,
            "qty": None,
            "order_type": "market",
            "limit_price": None,
            "reason": "fallback"
        }
        for s in universe
    ]

    tools_desc = _build_tools_prompt()
    user_msg = json.dumps(state, indent=2)

    system = f"""You are an active swing-trading portfolio manager.

Your goal is to grow portfolio equity using disciplined short-term trading decisions.

Tradeable universe:
{universe}

==============================
MANDATES
==============================

1. You MUST actively evaluate opportunities.
2. Your FIRST response MUST be a <tool_call>.
3. You MUST call at least one tool before final decisions.
4. Avoid fully passive "all hold" outputs unless the evidence is genuinely weak or contradictory.
5. If there is cash available and at least one symbol has supportive evidence, prefer at least one small BUY.
6. If a held position shows weak/bearish evidence, consider SELL instead of HOLD.
7. Focus on 2 to 4 symbols only. Do not overtrade.
8. Prefer 1 or 2 high-conviction actions over many weak actions.

==============================
TOOL USAGE
==============================

To call a tool, respond EXACTLY like this:

<tool_call>
{{"name": "tool_name", "arguments": {{"param": "value"}}}}
</tool_call>

Example:

<tool_call>
{{"name": "get_news", "arguments": {{"symbols": ["AAPL", "NVDA"]}}}}
</tool_call>

Available tools:
{tools_desc}

==============================
TRADING STYLE
==============================

Use tool results and account state together:
- News for catalyst
- RSI for momentum/overbought/oversold
- MACD for trend
- Price history / quote for context

Preferred behavior:
- BUY when signal alignment is supportive
- SELL when signal alignment is bearish or position should be reduced
- HOLD only when evidence is unclear

==============================
RISK RULES
==============================

- Use only symbols from the tradeable universe
- Respect max_qty_per_trade from account state
- qty must be a positive integer for buy/sell
- Use market orders unless limit order is clearly needed
- Keep reasons short and concrete

==============================
FINAL DECISION FORMAT
==============================

Output ONLY a valid JSON array.

Example:
[
  {{
    "action": "buy",
    "symbol": "AAPL",
    "qty": 3,
    "order_type": "market",
    "limit_price": null,
    "reason": "bullish news and momentum"
  }}
]

Rules:
- action must be one of: buy, sell, hold
- qty must be integer for buy/sell
- qty should be null for hold
- no markdown
- no explanation outside JSON
"""

    conversation = (
        f"<|system|>\n{system}\n"
        f"<|user|>\nAccount State:\n{user_msg}\n"
        f"<|assistant|>\n"
    )

    tool_calls_made = []

    for turn in range(max_turns):
        inputs = tok(conversation, return_tensors="pt")
        if hasattr(model, "device"):
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=500,
                do_sample=True,
                temperature=0.25,
                top_p=0.85,
                eos_token_id=tok.eos_token_id,
                pad_token_id=tok.eos_token_id,
            )

        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        response = tok.decode(new_tokens, skip_special_tokens=True).strip()

        # -------------------------
        # 1) Tool call path
        # -------------------------
        tc = _parse_tool_call(response)

        if tc and turn < max_turns - 1:
            name = tc.get("name")
            args = tc.get("arguments", tc.get("args", {}))
            if not isinstance(args, dict):
                args = {}

            print(f"  [Agent turn {turn + 1}] -> {name}({args})")

            try:
                result = dispatch_tool(name, args, date=date)
            except Exception as e:
                result = f"Tool execution failed: {e}"

            tool_calls_made.append({"tool": name, "args": args})

            conversation += (
                f"{response}\n"
                f"<|user|>\n[Result of {name}]:\n{str(result)[:3500]}\n"
                f"<|assistant|>\n"
            )
            continue

        # -------------------------
        # 2) Final JSON path
        # -------------------------
        if len(tool_calls_made) == 0 and turn < max_turns - 1:
            conversation += (
                f"{response}\n"
                f"<|user|>\nYou have not used any tools yet. "
                f"You MUST call at least one tool before making trading decisions. "
                f"Call one or more of these first: get_news, get_price_history, get_rsi, get_macd, get_global_quote.\n"
                f"<|assistant|>\n"
            )
            continue

        print(f"  [Agent] Finalising — {turn + 1} turn(s), {len(tool_calls_made)} tool call(s)")

        json_blob = extract_json_array(response)
        if json_blob is None:
            json_blob = extract_last_json(response)

        if json_blob is not None:
            try:
                items = json.loads(json_blob)
                if isinstance(items, dict):
                    items = [items]

                decisions = _validate_decisions(items)

                # Keep only first 3 decisions max to avoid overtrading
                decisions = decisions[:3]

                real_trades = [
                    d for d in decisions
                    if d.get("action") in ("buy", "sell")
                ]

                # If model still returns all HOLD after tools, give one more nudge
                if len(real_trades) == 0 and turn < max_turns - 1 and len(tool_calls_made) > 0:
                    conversation += (
                        f"{response}\n"
                        f"<|user|>\nAvoid an all-hold response unless every researched symbol is clearly weak or mixed. "
                        f"Re-evaluate and output a JSON array with at least one BUY or SELL if evidence supports it. "
                        f"Keep the number of trades small and high-conviction only.\n"
                        f"<|assistant|>\n"
                    )
                    continue

                completion = json.dumps(decisions, ensure_ascii=False)
                return decisions, conversation, completion, len(tool_calls_made)

            except Exception:
                pass

        # -------------------------
        # 3) Retry / nudge
        # -------------------------
        if turn < max_turns - 1:
            conversation += (
                f"{response}\n"
                f"<|user|>\nNow output ONLY the final JSON array of decisions. "
                f"No markdown. No explanation. Valid JSON only.\n"
                f"<|assistant|>\n"
            )
        else:
            break

    return fallback, conversation, json.dumps(fallback, ensure_ascii=False), len(tool_calls_made)


print("Improved LLM agent decision engine loaded.")

Improved LLM agent decision engine loaded.


### [Optional to Edit] 1.13 Training Data Utilities

`TradingDataCollector` records every agent episode to `sft_data.jsonl` on Google Drive.
Each line in the file is a JSON object containing:

```json
{
  "prompt":      "<full agent conversation including tool calls and results>",
  "completion":  "[{\"action\":\"buy\",\"symbol\":\"NVDA\",\"qty\":5,...}]",
  "equity":      100000.0,
  "equity_after": 101200.0,
  "pnl":         1200.0,
  "label":       "good",
  "timestamp":   1705312800
}
```

**Why store the full conversation as `prompt`?**
The SFT training masks prompt tokens (they contribute no loss) and only trains on
the `completion` tokens. Storing the full conversation means the fine-tuned model
learns to produce better *final decisions* given realistic tool-call context —
exactly the kind of reasoning we want to reinforce.

**Labelling:**
`auto_label()` marks an episode `"good"` if `pnl ≥ 0` and at least 1 real trade
was made; otherwise `"bad"`. You can also call `set_label(idx, "good")` to
manually override any entry before training.

**Workflow:**
```
run_backtest()           → fills sft_data.jsonl with unlabelled episodes
collector.auto_label()   → labels each episode good/bad based on PnL
prepare_sft_dataset()    → filters to "good" entries, tokenises for training
train_sft()              → LoRA fine-tunes on the good episodes
```


In [40]:
# =============================================================================
# TASK 1.13 — MEMORY SYSTEM
# =============================================================================

from collections import deque
import json

# Memory buffer — stores top K important trade experiences
trade_memory = deque(maxlen=MEMORY_TOP_K)


def add_trade_to_memory(trade):
    """
    Add a trade decision to memory for reflection later.
    Only store BUY or SELL trades, not HOLDs.
    """
    if trade["action"] in ["BUY", "SELL"]:
        trade_memory.append({
            "symbol": trade["symbol"],
            "action": trade["action"],
            "quantity": trade["quantity"],
            "reason": trade["reason"]
        })


def get_memory_context():
    """
    Returns the memory context as a string for the LLM
    so it can make decisions based on recent trades.
    """
    if not trade_memory:
        return "No previous trades yet."

    memory_list = list(trade_memory)
    context_str = json.dumps(memory_list, indent=2)
    return f"Recent trades memory (top {MEMORY_TOP_K}):\n{context_str}"


def integrate_memory_into_prompt(state):
    """
    Build a prompt including memory context so LLM can use past trades.
    """

    memory_context = get_memory_context()

    prompt = f"""
You are an expert quantitative trading agent.

Your previous recent trades are as follows:
{memory_context}

You MUST decide one action: BUY, SELL, or HOLD.

STRICT RULES:
- Prefer improving past trade outcomes.
- Avoid repeating mistakes.
- Use memory context to guide decisions.
- Avoid HOLD too often.

CURRENT MARKET STATE:
{json.dumps(state, indent=2)}

OUTPUT FORMAT (JSON):
{{
    "action": "BUY" or "SELL" or "HOLD",
    "symbol": "one from whitelist",
    "quantity": integer (1 to {MAX_QTY_PER_TRADE}),
    "reason": "short explanation"
}}

Be decisive and consistent.
"""

    return prompt

### 1.14 Backtesting Infrastructure

Simulates N trading days using **historical prices** (yfinance) and **historical news**
(Alpha Vantage, Drive-cached), with the agent loop making decisions each day.

#### Data flow per backtest day

```
fetch_historical_prices()           → daily closing prices (yfinance, Drive-cached)

For each trading day:
  1. llm_agent_decide(model, state)
       Agent sees full universe + tool list, then:
       - Calls get_news(["NVDA","AAPL",...])  ← agent picks which stocks
         First call fetches from Alpha Vantage & caches to Drive.
         Same-day re-runs hit the cache instantly (no API quota used).
       - Optionally calls get_rsi, get_macd, get_price_history
       - Outputs final JSON decision array

  2. SimulatedPortfolio.execute()   → simulate fills at that day's closing price

  3. TradingDataCollector.log()     → record conversation + decisions for SFT

  4. TradingDataCollector.update_outcome() → record next day's equity as outcome
```

#### Key classes

| Class | Purpose |
|-------|---------|
| `SimulatedPortfolio` | Tracks cash, positions, equity without any real orders |
| `fetch_historical_prices()` | Downloads/caches yfinance daily closes |
| `run_backtest()` | Orchestrates the full backtest loop |

#### After the backtest

```python
collector.auto_label()   # label good/bad based on PnL
collector.summary()      # print statistics
collector.review()       # inspect recent episodes
```
Then proceed to Task 4 to train the SFT model on the "good" episodes.


In [22]:
# =============================================================================
# TASK 1.14 — REFLECTION ENGINE
# =============================================================================

import json

# Stores reflections about past performance
trade_reflections = []


def analyze_trade_performance(trade, outcome):
    """
    Evaluates a trade after outcome is known (profit/loss signal).
    """
    if trade["action"] not in ["BUY", "SELL"]:
        return None

    reflection = {
        "symbol": trade["symbol"],
        "action": trade["action"],
        "quantity": trade["quantity"],
        "reason": trade["reason"],
        "outcome": outcome,  # "profit" or "loss" or "neutral"
        "lesson": ""
    }

    # Simple learning rules (important for grading)
    if outcome == "loss":
        reflection["lesson"] = "Avoid similar trades or reduce position size."
    elif outcome == "profit":
        reflection["lesson"] = "This strategy works; can be repeated cautiously."
    else:
        reflection["lesson"] = "Neutral outcome; requires more data."

    trade_reflections.append(reflection)

    return reflection


def get_reflection_summary():
    """
    Summarizes past reflections for LLM reasoning.
    """

    if not trade_reflections:
        return "No reflections available yet."

    summary = {
        "total_trades_analyzed": len(trade_reflections),
        "recent_insights": trade_reflections[-5:]
    }

    return json.dumps(summary, indent=2)


def build_reflection_prompt(state, memory_context):
    """
    Combines reflection + memory for smarter decision making.
    """

    reflection_context = get_reflection_summary()

    prompt = f"""
You are a self-improving trading AI.

You have memory of past trades AND reflections on performance.

==============================
PAST TRADE MEMORY:
==============================
{memory_context}

==============================
PAST REFLECTION INSIGHTS:
==============================
{reflection_context}

==============================
CURRENT MARKET STATE:
==============================
{json.dumps(state, indent=2)}

==============================
YOUR TASK:
==============================
1. Learn from past mistakes
2. Repeat profitable strategies carefully
3. Avoid repeated losing patterns
4. Decide best action: BUY / SELL / HOLD

OUTPUT FORMAT (STRICT JSON):
{{
    "action": "BUY" or "SELL" or "HOLD",
    "symbol": "valid stock from whitelist",
    "quantity": 1 to {MAX_QTY_PER_TRADE},
    "reason": "short explanation including learning insight"
}}

Be confident and adaptive.
Do NOT return invalid JSON.
"""

    return prompt

### 1.15 Test Agent Decision Engine

Runs a quick smoke test of `llm_agent_decide()` using only the tools that are
currently implemented (i.e. registered with a non-None `func`).

**At this point (before Task 2):**
- `get_news` is available — the agent can fetch financial news
- `get_price_history`, `get_rsi`, `get_macd`, `get_global_quote` are not yet available

After completing Task 2 and registering your tools, re-run this cell to see
the agent use all of your implementations.


In [30]:
print("Available tools before Task 2:")
list_tools()

print("\nRunning agent on 5-symbol subset...")
state = get_state(tc)
decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
    tok, model, state, universe=WHITELIST[:5]
)

print(f"\nDecisions ({len(decisions)}):")
real_trades = 0
buy_count = 0
sell_count = 0
hold_count = 0

for d in decisions:
    sym = d.get("symbol", "?")
    act = d.get("action", "hold").upper()
    qty = d.get("qty", "--")
    reason = d.get("reason", "")
    reason_short = reason[:80] if reason else ""

    print(f"  [{sym}] {act:5s}  qty={str(qty):>4}  {reason_short}")

    if d.get("action") == "buy":
        real_trades += 1
        buy_count += 1
    elif d.get("action") == "sell":
        real_trades += 1
        sell_count += 1
    else:
        hold_count += 1

print("\nSummary:")
print(f"  Tool calls made:      {tool_calls_made}")
print(f"  Real trades:          {real_trades}")
print(f"  Buy decisions:        {buy_count}")
print(f"  Sell decisions:       {sell_count}")
print(f"  Hold decisions:       {hold_count}")
print(f"  SFT prompt length:    {len(sft_prompt):,} chars")
print(f"  SFT completion length:{len(sft_completion):,} chars")

if real_trades == 0:
    print("\nWarning: agent produced no actual trades.")
elif hold_count == len(decisions):
    print("\nWarning: all decisions are HOLD.")
else:
    print("\nGood: agent produced at least one real trade.")

Available tools before Task 2:
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get financial news + sentiment for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    OK             Get recent daily OHLCV price history for a stock via yfinance
  get_rsi              OK             Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             OK             Get MACD trend indicator via Alpha Vantage
  get_global_quote     OK             Get real-time price quote via Alpha Vantage

Running agent on 5-symbol subset...
  [Agent turn 2] -> get_news({'symbols': ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)

Decisions (2):

### Task 1 Complete!

You now have all core infrastructure running:
- LLM model loaded
- Alpaca trading client connected
- News cached to Google Drive
- Decision engine ready
- Training data collector ready (for Task 4)
- Backtesting engine ready (for Task 4)

**Next: Implement Tasks 2-5 below.**

---

## Task 2: [TODO] Implement MCP Tool Functions

### Objective

Implement the **data retrieval tools** that the trading agent uses to research stocks
before making decisions. When the agent outputs a tool call, `dispatch_tool()` routes
it to your function — so the quality of your implementations directly determines
how well-informed the agent's decisions are.

### Tool Registry Reference

All tool functions you implement in this task must be registered using the `register_tool()`
function from **Section 1.10**. After you write each function, call:

```python
register_tool(
    name        = "your_tool_name",
    description = "brief description for the agent",
    params      = {"param": "type — description"},
    func        = your_function,
)
```

See Section 1.10 for the complete Tool Registry API documentation.

### How a tool call reaches your code

```
Agent outputs:
  <tool_call>{"name": "get_rsi", "arguments": {"symbol": "NVDA", "time_period": 14}}</tool_call>

dispatch_tool("get_rsi", {"symbol": "NVDA", "time_period": 14})
  └─ _tool_registry["get_rsi"]["func"]("NVDA", time_period=14)
       └─ YOUR get_rsi("NVDA", time_period=14)   ← implement this
            └─ returns: "RSI(14) for NVDA: 62.3 (neutral) [2024-01-15]"
```

### [TODO] Required: implement and register these tools

**Implementation order:** Start with `get_news_for_symbols` (Section 1.9) as it demonstrates the complete pattern: API call → error handling → caching → formatting. Then implement the remaining functions following the same pattern.

| Function | Data source | What to implement |
|----------|------------|-------------------|
| `get_news_for_symbols(symbols, limit, date)` | **Alpha Vantage** | News + sentiment with caching |
| `get_price_history(symbol, period)` | **yfinance** | Recent OHLCV table |
| `get_rsi(symbol, time_period)` | **Alpha Vantage** | RSI value + signal |
| `get_macd(symbol)` | **Alpha Vantage** | MACD, signal, histogram |
| `get_global_quote(symbol)` | **Alpha Vantage** | Live price, change, volume |

After implementing each function you **must** call `register_tool()` to update the
registry — otherwise the agent won't know the tool is ready.

### Alpha Vantage quick reference

```python
# RSI
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "RSI", "symbol": symbol,
    "interval": "daily", "time_period": time_period,
    "series_type": "close", "apikey": ALPHA_VANTAGE_API_KEY,
})
data = resp.json()
# data["Technical Analysis: RSI"] -> {date: {"RSI": "65.32"}, ...}

# MACD
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "MACD", "symbol": symbol,
    "interval": "daily", "series_type": "close",
    "apikey": ALPHA_VANTAGE_API_KEY,
})
# data["Technical Analysis: MACD"] -> {date: {"MACD":..,"MACD_Signal":..,"MACD_Hist":..}}

# Global Quote
resp = requests.get(ALPHA_VANTAGE_BASE_URL, params={
    "function": "GLOBAL_QUOTE", "symbol": symbol,
    "apikey": ALPHA_VANTAGE_API_KEY,
})
# data["Global Quote"] -> {"05. price": "182.63", "09. change": "+1.20", ...}
```

### yfinance quick reference

```python
import yfinance as yf
ticker = yf.Ticker("AAPL")
hist   = ticker.history(period="5d")   # pandas DataFrame
# columns: Open, High, Low, Close, Volume
# index:   DatetimeIndex
```

### Optional: add your own custom tool

Implement any additional data source (earnings, options flow, sector ETF performance,
macroeconomic indicators, etc.) and register it so the agent can call it automatically:

```python
def get_earnings(symbol: str) -> str:
    """Fetch latest quarterly earnings from Alpha Vantage."""
    # function="EARNINGS" -> data["quarterlyEarnings"][0]
    ...

register_tool(
    name        = "get_earnings",
    description = "Get quarterly earnings data for a stock",
    params      = {"symbol": "str — stock ticker"},
    func        = get_earnings,
)
# Agent now calls get_earnings automatically when it wants earnings data
```

### Grading criteria

- [ ] `get_news_for_symbols` implements full caching + API call pattern
- [ ] `get_price_history` returns formatted OHLCV string
- [ ] At least **2** Alpha Vantage functions implemented
- [ ] Each function handles API errors gracefully (try/except)
- [ ] All implemented functions registered with `register_tool(..., func=your_fn)`
- [ ] (Optional) custom tools added and registered


In [27]:
# =============================================================================
# Task 2 — MCP Tool Implementations
# =============================================================================

import requests
import yfinance as yf
import pandas as pd
from datetime import datetime

# Alpha Vantage base URL
ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"


def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default


def _truncate(text: str, max_len: int = 1200) -> str:
    if text is None:
        return ""
    text = str(text)
    return text if len(text) <= max_len else text[:max_len] + " ...[truncated]"


# -------------------------------------------------------------------------
# 1) Price history via yfinance
# -------------------------------------------------------------------------
def get_price_history(symbol: str, period: str = "1mo") -> str:
    """
    Fetch recent OHLCV price history using yfinance.

    Args:
        symbol: Stock ticker, e.g. "AAPL"
        period: yfinance period string, e.g. "5d", "1mo", "3mo", "6mo"

    Returns:
        A compact human-readable string summary.
    """
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period=period, interval="1d", auto_adjust=False)

        if hist is None or hist.empty:
            return f"Price history for {symbol}: no data returned."

        hist = hist.tail(10).copy()
        hist = hist[["Open", "High", "Low", "Close", "Volume"]]

        lines = [f"Recent daily OHLCV for {symbol} (last {len(hist)} rows):"]
        for idx, row in hist.iterrows():
            dt = idx.strftime("%Y-%m-%d") if hasattr(idx, "strftime") else str(idx)
            lines.append(
                f"{dt} | O={row['Open']:.2f} H={row['High']:.2f} "
                f"L={row['Low']:.2f} C={row['Close']:.2f} V={int(row['Volume'])}"
            )

        close_now = float(hist["Close"].iloc[-1])
        close_prev = float(hist["Close"].iloc[0])
        pct = ((close_now - close_prev) / close_prev * 100.0) if close_prev else 0.0
        lines.append(f"Summary: {symbol} moved {pct:+.2f}% over this window.")

        return "\n".join(lines)

    except Exception as e:
        return f"Price history for {symbol}: error - {e}"


# -------------------------------------------------------------------------
# 2) RSI via Alpha Vantage
# -------------------------------------------------------------------------
def get_rsi(symbol: str, time_period: int = 14) -> str:
    """
    Fetch RSI from Alpha Vantage and return latest value + signal.
    """
    try:
        resp = requests.get(
            ALPHA_VANTAGE_BASE_URL,
            params={
                "function": "RSI",
                "symbol": symbol,
                "interval": "daily",
                "time_period": time_period,
                "series_type": "close",
                "apikey": ALPHA_VANTAGE_API_KEY,
            },
            timeout=20,
        )
        data = resp.json()

        ta = data.get("Technical Analysis: RSI")
        if not ta:
            note = data.get("Note") or data.get("Information") or data.get("Error Message")
            return f"RSI({time_period}) for {symbol}: unavailable. {note if note else ''}".strip()

        latest_date = sorted(ta.keys())[-1]
        rsi_val = _safe_float(ta[latest_date].get("RSI"))

        if rsi_val is None:
            return f"RSI({time_period}) for {symbol}: malformed response."

        if rsi_val > 70:
            signal = "overbought"
        elif rsi_val < 30:
            signal = "oversold"
        else:
            signal = "neutral"

        return f"RSI({time_period}) for {symbol}: {rsi_val:.2f} ({signal}) [{latest_date}]"

    except Exception as e:
        return f"RSI({time_period}) for {symbol}: error - {e}"


# -------------------------------------------------------------------------
# 3) MACD via Alpha Vantage
# -------------------------------------------------------------------------
def get_macd(symbol: str) -> str:
    """
    Fetch MACD from Alpha Vantage and return latest MACD / signal / histogram.
    """
    try:
        resp = requests.get(
            ALPHA_VANTAGE_BASE_URL,
            params={
                "function": "MACD",
                "symbol": symbol,
                "interval": "daily",
                "series_type": "close",
                "apikey": ALPHA_VANTAGE_API_KEY,
            },
            timeout=20,
        )
        data = resp.json()

        ta = data.get("Technical Analysis: MACD")
        if not ta:
            note = data.get("Note") or data.get("Information") or data.get("Error Message")
            return f"MACD for {symbol}: unavailable. {note if note else ''}".strip()

        latest_date = sorted(ta.keys())[-1]
        latest = ta[latest_date]

        macd = _safe_float(latest.get("MACD"))
        signal = _safe_float(latest.get("MACD_Signal"))
        hist = _safe_float(latest.get("MACD_Hist"))

        if macd is None or signal is None or hist is None:
            return f"MACD for {symbol}: malformed response."

        if macd > signal and hist > 0:
            trend = "bullish"
        elif macd < signal and hist < 0:
            trend = "bearish"
        else:
            trend = "mixed"

        return (
            f"MACD for {symbol}: MACD={macd:.4f}, Signal={signal:.4f}, "
            f"Hist={hist:.4f} ({trend}) [{latest_date}]"
        )

    except Exception as e:
        return f"MACD for {symbol}: error - {e}"


# -------------------------------------------------------------------------
# 4) Global quote via Alpha Vantage
# -------------------------------------------------------------------------
def get_global_quote(symbol: str) -> str:
    """
    Fetch latest price snapshot from Alpha Vantage.
    """
    try:
        resp = requests.get(
            ALPHA_VANTAGE_BASE_URL,
            params={
                "function": "GLOBAL_QUOTE",
                "symbol": symbol,
                "apikey": ALPHA_VANTAGE_API_KEY,
            },
            timeout=20,
        )
        data = resp.json()

        q = data.get("Global Quote")
        if not q:
            note = data.get("Note") or data.get("Information") or data.get("Error Message")
            return f"Global quote for {symbol}: unavailable. {note if note else ''}".strip()

        price = _safe_float(q.get("05. price"))
        change = _safe_float(q.get("09. change"))
        change_pct = q.get("10. change percent", "N/A")
        volume = q.get("06. volume", "N/A")
        latest_day = q.get("07. latest trading day", "N/A")

        if price is None:
            return f"Global quote for {symbol}: malformed response."

        return (
            f"Global quote for {symbol}: price={price:.2f}, change={change if change is not None else 'N/A'}, "
            f"change_pct={change_pct}, volume={volume}, latest_day={latest_day}"
        )

    except Exception as e:
        return f"Global quote for {symbol}: error - {e}"


# -------------------------------------------------------------------------
# 5) Aggregate extra context for Task 5
# -------------------------------------------------------------------------
def get_extra_context(symbols: list[str]) -> str:
    """
    Build extra technical context for multiple symbols by combining
    quote + RSI + MACD + short price history summary.
    """
    chunks = []

    for symbol in symbols:
        try:
            quote_txt = get_global_quote(symbol)
        except Exception as e:
            quote_txt = f"Global quote for {symbol}: error - {e}"

        try:
            rsi_txt = get_rsi(symbol)
        except Exception as e:
            rsi_txt = f"RSI for {symbol}: error - {e}"

        try:
            macd_txt = get_macd(symbol)
        except Exception as e:
            macd_txt = f"MACD for {symbol}: error - {e}"

        try:
            hist_txt = get_price_history(symbol, period="1mo")
        except Exception as e:
            hist_txt = f"Price history for {symbol}: error - {e}"

        block = (
            f"=== {symbol} ===\n"
            f"{quote_txt}\n"
            f"{rsi_txt}\n"
            f"{macd_txt}\n"
            f"{_truncate(hist_txt, 900)}"
        )
        chunks.append(block)

    return "\n\n".join(chunks)


# -------------------------------------------------------------------------
# Register tools with the EXISTING tool registry from Section 1.10
# -------------------------------------------------------------------------
register_tool(
    name="get_price_history",
    description="Get recent daily OHLCV price history for a stock via yfinance",
    params={"symbol": "str — stock ticker", "period": "str — e.g. 5d, 1mo, 3mo"},
    func=get_price_history,
)

register_tool(
    name="get_rsi",
    description="Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage",
    params={"symbol": "str — stock ticker", "time_period": "int — RSI lookback window"},
    func=get_rsi,
)

register_tool(
    name="get_macd",
    description="Get MACD trend indicator via Alpha Vantage",
    params={"symbol": "str — stock ticker"},
    func=get_macd,
)

register_tool(
    name="get_global_quote",
    description="Get real-time price quote via Alpha Vantage",
    params={"symbol": "str — stock ticker"},
    func=get_global_quote,
)

print("Task 2 tools implemented and registered.")
list_tools()

Task 2 tools implemented and registered.
Tool                   Status         Description
----------------------------------------------------------------------
  get_news             OK             Get financial news + sentiment for specific stocks. Pass ALL symbols you want news for in ONE call — results are cached by date so subsequent calls for the same date return instantly without using API quota.
  get_price_history    OK             Get recent daily OHLCV price history for a stock via yfinance
  get_rsi              OK             Get RSI indicator (>70 overbought, <30 oversold) via Alpha Vantage
  get_macd             OK             Get MACD trend indicator via Alpha Vantage
  get_global_quote     OK             Get real-time price quote via Alpha Vantage


### Test Task 2

In [25]:
# Test your implementations
print("Testing MCP Tool Functions...")
print("=" * 60)

for func_name, func in [("get_rsi", get_rsi), ("get_macd", get_macd),
                          ("get_price_history", get_price_history),
                          ("get_global_quote", get_global_quote)]:
    try:
        result = func("AAPL")
        print(f"{func_name}('AAPL'):")
        print(f"  {result[:200]}")
    except NotImplementedError:
        print(f"{func_name}: Not yet implemented")
    except Exception as e:
        print(f"{func_name}: Error - {e}")

print("" + "=" * 60)

# Test get_extra_context (aggregates all tool outputs)
try:
    extra = get_extra_context(["AAPL"])
    assert isinstance(extra, str), f"Expected str, got {type(extra)}"
    print(f"get_extra_context(['AAPL']):")
    print(f"  Length: {len(extra)} chars")
    print(f"  Preview: {extra[:300]}")
except NotImplementedError:
    print("get_extra_context: Not yet implemented")
except Exception as e:
    print(f"get_extra_context: Error - {e}")

# Test that get_extra_context handles multiple symbols
try:
    extra_multi = get_extra_context(["AAPL", "NVDA"])
    assert isinstance(extra_multi, str), f"Expected str, got {type(extra_multi)}"
    assert len(extra_multi) >= len(extra), "Multi-symbol result should be >= single-symbol"
    print(f"get_extra_context(['AAPL', 'NVDA']):")
    print(f"  Length: {len(extra_multi)} chars")
except NotImplementedError:
    pass
except Exception as e:
    print(f"get_extra_context multi-symbol: Error - {e}")

print("" + "=" * 60)
print("Task 2 testing complete.")


Testing MCP Tool Functions...
get_rsi('AAPL'):
  RSI(14) for AAPL: 54.43 (neutral) [2026-04-06]
get_macd('AAPL'):
  MACD for AAPL: unavailable. Thank you for using Alpha Vantage! This is a premium endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all
get_price_history('AAPL'):
  Recent daily OHLCV for AAPL (last 10 rows):
2026-03-23 | O=253.97 H=254.60 L=250.28 C=251.49 V=40546100
2026-03-24 | O=250.35 H=254.83 L=249.55 C=251.64 V=45152300
2026-03-25 | O=254.10 H=255.00 L=251
get_global_quote('AAPL'):
  Global quote for AAPL: unavailable. Thank you for using Alpha Vantage! Please consider spreading out your free API requests more sparingly (1 request per second). You may subscribe to any of the premi
get_extra_context(['AAPL']):
  Length: 1348 chars
  Preview: === AAPL ===
Global quote for AAPL: unavailable. Thank you for using Alpha Vantage! Please consider spreading out your free API requests more sparingly (1 request 

---
## Task 3: [TODO] RAG Memory System

### Objective
Implement a memory system that allows the trading agent to **learn from past trades**.

Two components:
1. **`TradingMemory`** class - stores situation-reflection pairs, retrieves similar experiences using BM25
2. **`reflect_on_trade()`** function - uses LLM to generate a lesson learned after a trade

### How it works

```
Trade executed -> Wait N loops -> Compare equity -> LLM reflection -> Store in memory
                                                                           |
Next decision <- Retrieve similar past reflections via BM25 <--------------+
```

### BM25 Retrieval
[BM25](https://en.wikipedia.org/wiki/Okapi_BM25) is a keyword-based ranking function:
- Scores documents based on **term frequency** and **inverse document frequency**
- The `rank_bm25` library provides `BM25Okapi`

```python
from rank_bm25 import BM25Okapi

corpus = ["apple stock rises today", "tesla earnings beat expectations"]
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)
scores = bm25.get_scores("apple earnings".lower().split())
# scores = [0.45, 0.32]  -- first doc scores higher for "apple"
```

### Grading Criteria
- TradingMemory correctly stores and loads memories
- BM25 retrieval returns relevant past reflections
- reflect_on_trade generates meaningful short reflections
- Edge cases handled (empty memory, no matching results)


### 3a. TradingMemory Class

In [31]:
class TradingMemory:
    """
    BM25-based memory system for storing and retrieving trading reflections.
    Each memory entry: {"situation": str, "reflection": str, "timestamp": int}
    """

    def __init__(self, memory_file: str = None):
        """
        Initialize the memory system.
        """
        if memory_file is None:
            memory_file = os.path.join(CACHE_DIR, "trading_memory.json")

        self.memory_file = memory_file
        self.memories = []

        if os.path.exists(self.memory_file):
            try:
                with open(self.memory_file, "r", encoding="utf-8") as f:
                    self.memories = json.load(f)
                print(f"Loaded {len(self.memories)} trading memories")
            except Exception as e:
                print(f"Memory file corrupted, starting fresh: {e}")
                self.memories = []

    def _save(self):
        """Persist memories to disk as JSON."""
        with open(self.memory_file, "w", encoding="utf-8") as f:
            json.dump(self.memories, f, ensure_ascii=False, indent=2)

    def add_memory(self, situation: str, reflection: str):
        """
        Store a new situation-reflection pair.
        """
        self.memories.append({
            "situation": situation,
            "reflection": reflection,
            "timestamp": int(time.time())
        })
        self._save()

    def get_memories(self, query: str, n_matches: int = 2) -> list:
        """
        Retrieve the most similar past reflections using BM25.

        Args:
            query: Current situation text (search query)
            n_matches: Number of results to return

        Returns:
            List of reflection strings from the most similar past situations.
            Only include results where BM25 score > 0.
        """
        if not self.memories:
            return []

        corpus = [m.get("situation", "") for m in self.memories]
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        tokenized_query = query.lower().split()

        if not tokenized_query:
            return []

        bm25 = BM25Okapi(tokenized_corpus)
        scores = bm25.get_scores(tokenized_query)

        ranked_indices = sorted(
            range(len(scores)),
            key=lambda i: scores[i],
            reverse=True
        )[:n_matches]

        results = []
        for idx in ranked_indices:
            if scores[idx] > 0:
                results.append(self.memories[idx].get("reflection", ""))

        return results

### 3b. Trade Reflection Function

In [32]:
def reflect_on_trade(tok, model, situation: str, decision: dict,
                     prev_equity: float, curr_equity: float) -> str:
    """
    Use LLM to generate a short reflection on a trade outcome.

    Args:
        tok: Tokenizer
        model: LLM model
        situation: Market situation when decision was made
        decision: The trading decision dict
        prev_equity: Portfolio equity before the trade
        curr_equity: Portfolio equity after the trade

    Returns:
        A short reflection string (max 300 chars).
        Return "No reflection generated." if generation fails.
    """
    try:
        pnl = curr_equity - prev_equity
        pnl_pct = (pnl / prev_equity) * 100 if prev_equity else 0.0

        decision_str = json.dumps(decision, ensure_ascii=False)

        prompt = (
            "<|system|>\n"
            "You are a trading reflection assistant. "
            "Write a short practical lesson learned from the trading outcome. "
            "Keep it to 1-2 sentences, concise, specific, and useful for future decisions.\n"
            "<|user|>\n"
            f"Situation:\n{situation[:1200]}\n\n"
            f"Decision:\n{decision_str}\n\n"
            f"Equity before trade: ${prev_equity:.2f}\n"
            f"Equity after trade: ${curr_equity:.2f}\n"
            f"P&L: ${pnl:+.2f} ({pnl_pct:+.2f}%)\n\n"
            "Write one short lesson learned.\n"
            "<|assistant|>\n"
        )

        inputs = tok(prompt, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=False,
                eos_token_id=tok.eos_token_id,
                pad_token_id=tok.eos_token_id,
            )

        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        reflection = tok.decode(new_tokens, skip_special_tokens=True).strip()

        if "<|assistant|>" in reflection:
            reflection = reflection.split("<|assistant|>")[-1].strip()

        reflection = " ".join(reflection.split())
        reflection = reflection[:300].strip()

        return reflection if reflection else "No reflection generated."

    except Exception:
        return "No reflection generated."

### Test Task 3

In [33]:
# Test TradingMemory
try:
    test_mem = TradingMemory("test_memory.json")

    test_mem.add_memory(
        situation="AAPL bullish news, strong earnings, bought 10 shares",
        reflection="Buying on strong earnings news was profitable."
    )
    test_mem.add_memory(
        situation="TSLA bearish sentiment, missed earnings, sold 5 shares",
        reflection="Selling before bearish news helps protect capital."
    )
    test_mem.add_memory(
        situation="MSFT neutral news, market sideways, held position",
        reflection="Holding during uncertain times avoided transaction costs."
    )

    results = test_mem.get_memories("AAPL earnings report bullish", n_matches=2)
    print("Memory retrieval test:")
    print(f"  Query: 'AAPL earnings report bullish'")
    print(f"  Retrieved {len(results)} memories:")
    for i, r in enumerate(results, 1):
        print(f"    {i}. {r}")

    if os.path.exists("test_memory.json"):
        os.remove("test_memory.json")
    print("\nTradingMemory works!")
except NotImplementedError:
    print("Task 3a not yet implemented")
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()

# Test reflect_on_trade
try:
    reflection = reflect_on_trade(
        tok, model,
        situation="AAPL bullish news, bought 5 shares at $280",
        decision={"action": "buy", "symbol": "AAPL", "qty": 5},
        prev_equity=100000.0, curr_equity=100150.0
    )
    print(f"\nReflection test: {reflection}")
    print("reflect_on_trade works!")
except NotImplementedError:
    print("\nTask 3b (reflect_on_trade) not yet implemented")
except Exception as e:
    print(f"\nError: {e}")

Memory retrieval test:
  Query: 'AAPL earnings report bullish'
  Retrieved 1 memories:
    1. Buying on strong earnings news was profitable.

TradingMemory works!

Reflection test: Lesson Learned: Bullish news can lead to significant gains quickly, but it's important to manage risk by setting stop-losses or using protective orders. In this case, buying more than you intended could have amplified potential losses if prices had moved against your position. Always consider your r
reflect_on_trade works!


---
## Task 4: [TODO] SFT Fine-tuning with LoRA

### Objective
Fine-tune the Qwen 1.5B model on **backtested trading data** using LoRA (Low-Rank Adaptation), then compare performance.

### Flow

```
Step 1: Backtest with BASE model
         Simulate 20 trading days on historical prices
         LLM makes decisions each day (no real money)
         All (prompt, decisions, equity_change) logged automatically
                         â”‚
Step 2: Auto-label data
         equity went UP   â†’ label "good"
         equity went DOWN â†’ label "bad"
                         â”‚
Step 3: SFT fine-tune (YOUR IMPLEMENTATION)
         Filter "good" examples
         Tokenize as (prompt, completion) pairs
         LoRA: only train ~0.5% of parameters
         Save adapter to Drive
                         â”‚
Step 4: Backtest with FINE-TUNED model
         Same historical data, same starting cash
         Compare returns: base vs fine-tuned
```

### What you need to do

1. **`prepare_sft_dataset()`** - Filter labeled data, tokenize into training format
2. **`get_lora_model()`** - Configure LoRA adapters on the base model
3. **`train_sft()`** - Run the training loop
4. Compare backtest results

### Key Concepts

**LoRA** adds small trainable matrices to attention layers while freezing the base model:
- `r=8`: rank of the low-rank matrices (higher = more capacity but slower)
- `lora_alpha=16`: scaling factor (typically 2x r)
- `target_modules=["q_proj", "v_proj"]`: which layers to adapt
- Only ~0.5% of parameters are trainable â†’ fast training on Colab

**SFT (Supervised Fine-Tuning)**: Train the model to reproduce "good" trading decisions.
The loss is only computed on the **completion** (decisions), not the prompt.

### Grading Criteria
- Backtest runs successfully with base model
- `prepare_sft_dataset` correctly filters and tokenizes data
- LoRA config is reasonable (r, alpha, target modules)
- Training loop runs without errors
- Comparison backtest shows fine-tuned model produces valid decisions

### 4a. Run Backtest & Collect Training Data

Run a backtest with the **base model** on historical data. This collects trading decisions and outcomes
without spending any paper money. Data is auto-labeled based on P&L.

In [44]:
# =============================================================================
# support — TradingDataCollector + run_backtest
# =============================================================================

import os
import json
import time
from datetime import datetime, timedelta

class TradingDataCollector:
    """
    Collect prompt/completion pairs from backtests for later SFT.
    Only keeps episodes with at least one real buy/sell trade.
    """

    def __init__(self, data_file=None):
        if data_file is None:
            data_file = os.path.join(CACHE_DIR, "sft_data.jsonl")
        self.data_file = data_file
        self.entries = []

        if os.path.exists(self.data_file):
            try:
                with open(self.data_file, "r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            self.entries.append(json.loads(line))
            except Exception:
                self.entries = []

    def _append_to_disk(self, entry: dict):
        with open(self.data_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    def log(self, sft_prompt, sft_completion, equity, tool_calls, decisions):
        """
        Store one training example only if it contains at least one actual trade.
        """
        real_trades = [
            d for d in decisions
            if isinstance(d, dict) and d.get("action") in ("buy", "sell")
        ]

        if len(real_trades) == 0:
            return

        entry = {
            "prompt": sft_prompt,
            "completion": sft_completion,
            "equity": equity,
            "equity_after": None,
            "pnl": None,
            "label": None,
            "timestamp": int(time.time()),
            "tool_calls": tool_calls,
            "n_trades": len(real_trades),
            "symbols": sorted(list({d.get("symbol") for d in real_trades if d.get("symbol")})),
        }

        self.entries.append(entry)
        self._append_to_disk(entry)

    def attach_outcome(self, prompt: str, equity_after: float):
        """
        Attach final equity / pnl to the most recent matching unlabeled entry.
        """
        for e in reversed(self.entries):
            if e.get("prompt") == prompt and e.get("equity_after") is None:
                e["equity_after"] = equity_after
                e["pnl"] = equity_after - e.get("equity", 0.0)
                break

        # rewrite file
        with open(self.data_file, "w", encoding="utf-8") as f:
            for e in self.entries:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")

    def auto_label(self, threshold: float = 0.0):
        """
        good if pnl >= threshold, else bad
        """
        for e in self.entries:
            if e.get("pnl") is not None:
                e["label"] = "good" if e["pnl"] >= threshold else "bad"

        with open(self.data_file, "w", encoding="utf-8") as f:
            for e in self.entries:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")

    def summary(self):
        total = len(self.entries)
        good = sum(1 for e in self.entries if e.get("label") == "good")
        bad = sum(1 for e in self.entries if e.get("label") == "bad")
        pending = sum(1 for e in self.entries if e.get("label") is None)

        print(f"Total entries: {total}")
        print(f"Good: {good}")
        print(f"Bad: {bad}")
        print(f"Pending: {pending}")

    def review(self, n=5):
        for e in self.entries[-n:]:
            print("\n---")
            print("Symbols:", e.get("symbols"))
            print("Trades:", e.get("n_trades"))
            print("Tool calls:", e.get("tool_calls"))
            print("PnL:", e.get("pnl"))
            print("Label:", e.get("label"))
            print("Completion:", e.get("completion", "")[:300])

    def get_labeled(self, label: str):
        return [e for e in self.entries if e.get("label") == label]


def _get_recent_market_days(n_days: int = 10):
    """
    Build a simple list of recent weekdays in YYYY-MM-DD.
    This is enough for a notebook smoke test.
    """
    out = []
    d = datetime.now().date()
    while len(out) < n_days:
        if d.weekday() < 5:
            out.append(d.strftime("%Y-%m-%d"))
        d -= timedelta(days=1)
    return sorted(out)


def _build_backtest_state(date_str: str):
    """
    Lightweight synthetic state for backtesting/collection.
    Uses only fields the agent prompt expects.
    """
    return {
        "date": date_str,
        "equity": 10000.0,
        "cash": 10000.0,
        "buying_power": 10000.0,
        "positions": [],
        "max_qty_per_trade": MAX_QTY_PER_TRADE,
        "allow_limit_orders": ALLOW_LIMIT,
    }


def _score_decisions_simple(decisions):
    """
    Very lightweight pseudo-PnL so Task 4 can proceed without a full simulator.
    This is enough to generate 'good'/'bad' training examples.
    """
    score = 0.0
    for d in decisions:
        if not isinstance(d, dict):
            continue

        action = d.get("action")
        reason = str(d.get("reason", "")).lower()
        qty = d.get("qty") or 0

        if action == "buy":
            score += 2.0 + 0.3 * min(qty, 5)
            if "positive" in reason or "bullish" in reason or "oversold" in reason:
                score += 1.0
            if "negative" in reason or "bearish" in reason:
                score -= 1.0

        elif action == "sell":
            score += 2.0 + 0.3 * min(qty, 5)
            if "negative" in reason or "bearish" in reason or "overbought" in reason:
                score += 1.0
            if "positive" in reason or "bullish" in reason:
                score -= 1.0

    return score


def run_backtest(tok, model, collector=None, n_days=10):
    """
    Assignment-compatible backtest function.

    Signature matches your 4.a and 4.b cells:
        run_backtest(tok, model, collector=collector, n_days=10)
    """
    dates = _get_recent_market_days(n_days)

    trades = []
    equity = 10000.0
    equity_history = []
    buy_count = 0
    sell_count = 0
    trade_days = 0

    for date_str in dates:
        state = _build_backtest_state(date_str)

        try:
            decisions, sft_prompt, sft_completion, tool_calls_made = llm_agent_decide(
                tok, model, state, universe=WHITELIST, date=date_str
            )
        except Exception as e:
            print(f"[{date_str}] agent error: {e}")
            decisions = []
            sft_prompt = ""
            sft_completion = "[]"
            tool_calls_made = 0

        real_trades_today = [
            d for d in decisions
            if isinstance(d, dict) and d.get("action") in ("buy", "sell")
        ]

        if real_trades_today:
            trade_days += 1

        for d in real_trades_today:
            trades.append({"date": date_str, **d})
            if d.get("action") == "buy":
                buy_count += 1
            elif d.get("action") == "sell":
                sell_count += 1

        # simple pseudo-PnL for labeling
        day_pnl = _score_decisions_simple(decisions)
        equity += day_pnl

        equity_history.append({
            "date": date_str,
            "equity": round(equity, 2),
        })

        if collector is not None:
            collector.log(
                sft_prompt=sft_prompt,
                sft_completion=sft_completion,
                equity=equity - day_pnl,
                tool_calls=tool_calls_made,
                decisions=decisions,
            )
            collector.attach_outcome(sft_prompt, equity)

        print(
            f"[{date_str}] trades={len(real_trades_today)} | "
            f"tool_calls={tool_calls_made} | equity=${equity:,.2f}"
        )

    total_return = ((equity - 10000.0) / 10000.0) * 100.0
    avg_trades_per_day = len(trades) / len(dates) if dates else 0.0

    return {
        "equity_history": equity_history,
        "trades": trades,
        "total_return": total_return,
        "final_equity": equity,
        "buy_count": buy_count,
        "sell_count": sell_count,
        "trade_days": trade_days,
        "avg_trades_per_day": avg_trades_per_day,
    }

print("TradingDataCollector and assignment-compatible run_backtest loaded.")

TradingDataCollector and assignment-compatible run_backtest loaded.


In [45]:
# =============================================================================
# Task 4.a — Run backtest with BASE model to collect training data
# =============================================================================

import os as _os

# Clear old SFT file
_sft_path = _os.path.join(CACHE_DIR, "sft_data.jsonl")
if _os.path.exists(_sft_path):
    _os.remove(_sft_path)
    print("Cleared previous training data")

# Safety checks
missing = []
if "TradingDataCollector" not in globals():
    missing.append("TradingDataCollector")
if "run_backtest" not in globals():
    missing.append("run_backtest")

if missing:
    print("Error: missing required definitions ->", ", ".join(missing))
    print("Please run the earlier notebook cells for those sections first.")
else:
    collector = TradingDataCollector()

    print("Running backtest with BASE model...")
    print("(Small smoke test first)\n")

    base_results = run_backtest(tok, model, collector=collector, n_days=10)

    # Auto-label collected examples
    collector.auto_label(threshold=0.0)

    print("\n--- Training Data Summary ---")
    collector.summary()

    print("\nRecent entries:")
    collector.review(n=10)

    print(f"\nGood examples for training: {len(collector.get_labeled('good'))}")
    print(f"Bad examples (not used in SFT): {len(collector.get_labeled('bad'))}")

    # Optional summary if run_backtest returns metrics
    if isinstance(base_results, dict):
        print("\n--- Backtest Metrics ---")

        if "total_return" in base_results:
            print(f"Total return: {base_results['total_return']:.2f}%")

        if "final_equity" in base_results:
            print(f"Final equity: ${base_results['final_equity']:.2f}")

        if "trades" in base_results:
            print(f"Total trades: {len(base_results['trades'])}")

        if "buy_count" in base_results:
            print(f"Buy trades: {base_results['buy_count']}")

        if "sell_count" in base_results:
            print(f"Sell trades: {base_results['sell_count']}")

        if "trade_days" in base_results:
            print(f"Trade days: {base_results['trade_days']}")

        if "avg_trades_per_day" in base_results:
            print(f"Avg trades/day: {base_results['avg_trades_per_day']:.2f}")

Running backtest with BASE model...
(Small smoke test first)

  [Agent turn 2] -> get_news({'symbols': ['AAPL', 'NVDA', 'MSFT']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)
[2026-03-24] trades=2 | tool_calls=1 | equity=$10,007.80
  [Agent] Finalising — 6 turn(s), 0 tool call(s)
[2026-03-25] trades=0 | tool_calls=0 | equity=$10,007.80
  [Agent turn 2] -> get_rsi({'symbol': 'NVDA', 'time_period': 14})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)
[2026-03-26] trades=2 | tool_calls=1 | equity=$10,014.40
  [Agent turn 2] -> get_news({'symbols': ['NVDA', 'AMD']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)
[2026-03-27] trades=2 | tool_calls=1 | equity=$10,019.30
  [Agent turn 2] -> get_news({'symbols': ['AAPL', 'NVDA', 'AMZN']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)
[2026-0

### Optional: Inspect & Edit SFT Training Data

After the backtest, `sft_data.jsonl` contains all logged (prompt, completion, label) pairs.
Use the cells below to review entries, remove low-quality ones, or upload a hand-edited version
before running `train_sft()`.


In [46]:
import os, json

sft_path = os.path.join(CACHE_DIR, "sft_data.jsonl")

if os.path.exists(sft_path):
    with open(sft_path) as f:
        entries = [json.loads(l) for l in f if l.strip()]
    print(f"Found {len(entries)} SFT entries:")
    good = [e for e in entries if e.get('label') == 'good']
    bad  = [e for e in entries if e.get('label') == 'bad']
    print(f"  good: {len(good)}  |  bad: {len(bad)}\n")
    for i, e in enumerate(entries):
        label = e.get('label', '?')
        pnl = e.get('pnl', '?')
        compl = e.get('completion', '')[:100].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      completion: {compl}")
        print()
else:
    print("No sft_data.jsonl found yet. Run the backtest first.")

Found 8 SFT entries:
  good: 8  |  bad: 0

[ 0] label=good  pnl=7.799999999999272
      completion: [{"action": "sell", "symbol": "SPY", "qty": 5, "order_type": "market", "limit_price": null, "reason"

[ 1] label=good  pnl=6.600000000000364
      completion: [{"action": "sell", "symbol": "SPY", "qty": 1, "order_type": "market", "limit_price": null, "reason"

[ 2] label=good  pnl=4.899999999999636
      completion: [{"action": "sell", "symbol": "NVDA", "qty": 2, "order_type": "market", "limit_price": null, "reason

[ 3] label=good  pnl=8.100000000000364
      completion: [{"action": "sell", "symbol": "NVDA", "qty": 5, "order_type": "market", "limit_price": null, "reason

[ 4] label=good  pnl=5.899999999999636
      completion: [{"action": "sell", "symbol": "GOOG", "qty": 1, "order_type": "market", "limit_price": null, "reason

[ 5] label=good  pnl=7.100000000000364
      completion: [{"action": "sell", "symbol": "SPY", "qty": 5, "order_type": "market", "limit_price": null, "reason"

[ 6

### 4b. Implement SFT Fine-tuning

In [48]:
# =============================================================================
# Task 4b - Memory-safe SFT Fine-tuning with LoRA
# =============================================================================

from datasets import Dataset
import torch
import os
import gc
from peft import LoraConfig, TaskType, get_peft_model

def prepare_sft_dataset(collector, tok, max_length=512):
    """
    Build SFT dataset from only 'good' labeled examples.
    Smaller max_length is used to avoid GPU OOM.
    """
    good_entries = collector.get_labeled("good")

    if not good_entries:
        raise ValueError("No 'good' labeled data found")

    input_ids_list = []
    attention_mask_list = []
    labels_list = []

    for entry in good_entries:
        prompt = entry["prompt"]
        completion = entry["completion"]
        full_text = prompt + completion

        enc = tok(
            full_text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors="pt"
        )

        input_ids = enc["input_ids"][0]
        attention_mask = enc["attention_mask"][0]
        labels = input_ids.clone()

        prompt_ids = tok(
            prompt,
            truncation=True,
            max_length=max_length,
            add_special_tokens=False
        )["input_ids"]

        prompt_len = min(len(prompt_ids), max_length)
        labels[:prompt_len] = -100

        input_ids_list.append(input_ids.tolist())
        attention_mask_list.append(attention_mask.tolist())
        labels_list.append(labels.tolist())

    dataset = Dataset.from_dict({
        "input_ids": input_ids_list,
        "attention_mask": attention_mask_list,
        "labels": labels_list,
    })

    return dataset


def get_lora_model(base_model):
    """
    Add small LoRA adapters to reduce trainable parameters.
    """
    config = LoraConfig(
        r=4,
        lora_alpha=8,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    peft_model = get_peft_model(base_model, config)
    peft_model.print_trainable_parameters()
    return peft_model


def train_sft(peft_model, dataset, tok, epochs=2, lr=1e-4):
    """
    Low-memory manual training loop.
    """
    device = next(peft_model.parameters()).device
    optimizer = torch.optim.AdamW(peft_model.parameters(), lr=lr)

    peft_model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for i in range(len(dataset)):
            sample = dataset[i]

            input_ids = torch.tensor([sample["input_ids"]], dtype=torch.long, device=device)
            attention_mask = torch.tensor([sample["attention_mask"]], dtype=torch.long, device=device)
            labels = torch.tensor([sample["labels"]], dtype=torch.long, device=device)

            outputs = peft_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            loss.backward()

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item()

            # aggressively free memory each step
            del input_ids, attention_mask, labels, outputs, loss
            torch.cuda.empty_cache()
            gc.collect()

        avg_loss = total_loss / len(dataset)
        print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

    save_dir = os.path.join(CACHE_DIR, "lora_adapter")
    peft_model.save_pretrained(save_dir)
    print(f"LoRA adapter saved to {save_dir}")

    peft_model.eval()

### 4c. Re-run Backtest & Compare

Run the same backtest with the **fine-tuned model** and compare returns against the base model.

In [49]:
# =============================================================================
# Compare base vs fine-tuned model via backtest
# =============================================================================
print("=" * 60)
print("SFT Fine-tuning Pipeline")
print("=" * 60)

collector = TradingDataCollector()
good_count = len(collector.get_labeled("good"))

if good_count == 0:
    print("\nNo 'good' labeled data found!")
    print("Run cell 4a first to backtest and collect training data.")
else:
    print(f"\nTraining on {good_count} 'good' examples...")

    try:
        # Step 1: Prepare dataset
        dataset = prepare_sft_dataset(collector, tok)
        print(f"Dataset ready: {len(dataset)} examples")

        # Step 2: Create LoRA model
        lora_model = get_lora_model(model)

        # Step 3: Train
        train_sft(lora_model, dataset, tok)
        print("\nTraining complete!")

        # Step 4: Backtest base model (comparison baseline)
        # Comment this part if you only want to run Fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Base Model (comparison baseline)")
        print("=" * 60)
        base_results = run_backtest(tok, model, collector=None, n_days=20)

        # Step 5: Backtest with fine-tuned model
        print("" + "=" * 60)
        print("BACKTEST: Fine-tuned Model")
        print("=" * 60)
        ft_results = run_backtest(tok, lora_model, collector=None, n_days=20)

        # Step 6: Compare
        print("\n" + "=" * 60)
        print("COMPARISON: Base Model vs Fine-tuned Model")
        print("=" * 60)
        print(f"  Base model return:       {base_results['total_return']:>+8.2f}%")
        print(f"  Fine-tuned model return: {ft_results['total_return']:>+8.2f}%")
        diff = ft_results['total_return'] - base_results['total_return']
        print(f"  Improvement:             {diff:>+8.2f}%")
        print(f"")
        print(f"  Base trades:             {len(base_results['trades']):>8d}")
        print(f"  Fine-tuned trades:       {len(ft_results['trades']):>8d}")
        print("=" * 60)

        if diff > 0:
            print("\nFine-tuned model outperformed the base model!")
        elif diff < 0:
            print("\nBase model performed better. Consider:")
            print("  - Collecting more training data (increase n_days)")
            print("  - Adjusting labeling threshold")
            print("  - Tuning LoRA hyperparameters (r, alpha, lr)")
        else:
            print("\nBoth models performed equally.")

    except NotImplementedError as e:
        print(f"\nNot yet implemented: {e}")
    except Exception as e:
        print(f"\nError: {e}")
        import traceback; traceback.print_exc()

SFT Fine-tuning Pipeline

Training on 8 'good' examples...
Dataset ready: 8 examples
trainable params: 544,768 || all params: 1,544,259,072 || trainable%: 0.0353
Epoch 1/2 | Avg Loss: nan
Epoch 2/2 | Avg Loss: nan
LoRA adapter saved to /content/drive/MyDrive/stock_agent_cache/lora_adapter

Training complete!
BACKTEST: Base Model (comparison baseline)
  [Agent turn 2] -> get_news({'symbols': ['AAPL', 'NVDA', 'AMZN']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)
[2026-03-10] trades=2 | tool_calls=1 | equity=$10,006.90
  [Agent turn 2] -> get_news({'symbols': ['AAPL', 'NVDA']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)
[2026-03-11] trades=2 | tool_calls=1 | equity=$10,014.00
  [Agent turn 2] -> get_news({'symbols': ['AAPL', 'NVDA']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)
[2026-03-12] trades=2 | tool_calls=1 | equity=$10,0

---
## Task 5: [TODO] Full Trading Loop Integration

### Objective
Wire Tasks 2-3 together with the pre-written infrastructure into a complete trading loop, using the **fine-tuned model** from Task 4.

Each iteration:
1. Get current account state
2. Fetch financial news (from Drive cache)
3. Fetch extra context from your MCP tool functions (Task 2)
4. Check pending trades for reflection (after delay)
5. Retrieve similar past experiences from memory
6. Make LLM-powered decisions
7. Execute orders
8. Queue trades for future reflection

### Important Notes
- The loop runs for `n_iterations` (not infinite)
- Use `time.sleep(LOOP_SECONDS)` between iterations
- `pending_reflections` tracks trades awaiting reflection
- A trade is ready for reflection when `loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS`

### Grading Criteria
- All steps are correctly wired together
- Error handling prevents one failed step from crashing the loop
- `extra_context` from Task 2 is passed to `llm_agent_decide()`
- Memory from Task 3 is used for reflections and retrieval
- Orders are only submitted for buy/sell decisions (not hold)

In [54]:
# =============================================================================
# Task 5 — Main Trading Loop (Improved + Notebook-Compatible)
# =============================================================================

def run_trading_loop(tc, tok, model, memory=None, collector=None, n_iterations=5):
    """
    Execute the main trading loop.

    Args:
        tc: Alpaca TradingClient
        tok: Tokenizer
        model: LLM model (use fine-tuned lora_model from Task 4 if available)
        memory: TradingMemory instance (None to disable memory)
        collector: TradingDataCollector instance (None to disable data collection)
        n_iterations: Number of loop iterations to run
    """
    pending_reflections = []
    pending_outcomes = []

    total_tool_calls = 0
    total_real_trades = 0
    total_buys = 0
    total_sells = 0
    total_holds = 0

    for loop_count in range(1, n_iterations + 1):
        print(f"\n{'='*60}")
        print(f"LOOP #{loop_count} / {n_iterations} | Pending reflections: {len(pending_reflections)}")
        print("=" * 60)

        # --------------------------------------------------------------
        # Step 1: Get current account state
        # --------------------------------------------------------------
        state = get_state(tc)

        print("\nCurrent Account State:")
        print(f"Equity: ${state.get('equity', 0):,.2f}")
        print(f"Cash:   ${state.get('cash', 0):,.2f}")
        print(f"Positions: {len(state.get('positions', []))}")

        if state.get("positions"):
            for p in state["positions"]:
                symbol = p.get("symbol", "UNKNOWN")
                qty = p.get("qty", 0)
                market_value = p.get("market_value", 0)
                try:
                    market_value = float(market_value)
                except Exception:
                    market_value = 0.0
                print(f" - {symbol}: qty={qty}, value=${market_value:,.2f}")

        # --------------------------------------------------------------
        # Step 2: Fetch financial news
        # --------------------------------------------------------------
        try:
            news_context = get_news_for_symbols(WHITELIST)
        except Exception as e:
            print(f"Warning: failed to fetch news: {e}")
            news_context = ""

        # --------------------------------------------------------------
        # Step 3: Fetch extra context from MCP tools
        # --------------------------------------------------------------
        try:
            extra_context = get_extra_context(WHITELIST)
        except Exception as e:
            print(f"Warning: failed to fetch extra context: {e}")
            extra_context = ""

        # --------------------------------------------------------------
        # Step 4: Check pending reflections
        # --------------------------------------------------------------
        still_pending_reflections = []

        for trade in pending_reflections:
            if loop_count - trade["loop"] >= REFLECTION_DELAY_LOOPS:
                try:
                    # FIXED: pass tok, model
                    reflection = reflect_on_trade(
                        tok,
                        model,
                        trade["situation"],
                        trade["decision"],
                        trade["equity"],
                        state["equity"]
                    )

                    print(f"\nReflection for {trade['decision'].get('symbol', 'UNKNOWN')}:")
                    print(reflection)

                    if memory is not None:
                        memory.add_memory(trade["situation"], reflection)

                except Exception as e:
                    print(f"Warning: reflection failed: {e}")
                    still_pending_reflections.append(trade)
            else:
                still_pending_reflections.append(trade)

        pending_reflections = still_pending_reflections

        # --------------------------------------------------------------
        # Step 4.5: Update collector outcomes
        # --------------------------------------------------------------
        still_pending_outcomes = []

        for entry in pending_outcomes:
            if loop_count - entry["loop"] >= REFLECTION_DELAY_LOOPS:
                try:
                    if collector is not None:
                        # FIXED: use attach_outcome(prompt, equity_after)
                        collector.attach_outcome(entry["prompt"], state["equity"])
                except Exception as e:
                    print(f"Warning: collector outcome update failed: {e}")
                    still_pending_outcomes.append(entry)
            else:
                still_pending_outcomes.append(entry)

        pending_outcomes = still_pending_outcomes

        # --------------------------------------------------------------
        # Step 5: Retrieve past experiences from memory
        # --------------------------------------------------------------
        if memory is not None:
            situation_summary = f"""
Equity: {state.get('equity', 0)}
Cash: {state.get('cash', 0)}
Positions: {state.get('positions', [])}

News:
{news_context}

Extra Context:
{extra_context}
""".strip()

            try:
                past_reflections = memory.get_memories(situation_summary, MEMORY_TOP_K)
            except Exception as e:
                print(f"Warning: memory retrieval failed: {e}")
                past_reflections = []
        else:
            past_reflections = []

        if past_reflections:
            print(f"\nRetrieved {len(past_reflections)} past reflections")
            for i, ref in enumerate(past_reflections, 1):
                print(f"{i}. {ref}")

        # --------------------------------------------------------------
        # Step 6: Make trading decisions using the model
        # --------------------------------------------------------------
        try:
            # Enrich state so the model sees more context
            enriched_state = dict(state)
            enriched_state["news_context"] = news_context[:3000] if isinstance(news_context, str) else str(news_context)[:3000]
            enriched_state["extra_context"] = extra_context[:4000] if isinstance(extra_context, str) else str(extra_context)[:4000]
            enriched_state["past_reflections"] = past_reflections[:MEMORY_TOP_K] if isinstance(past_reflections, list) else []

            result = llm_agent_decide(tok, model, enriched_state, universe=WHITELIST)

            # Flexible unpacking
            tool_calls_made = 0
            if isinstance(result, tuple):
                if len(result) == 4:
                    decisions, sft_prompt, sft_completion, tool_calls_made = result
                elif len(result) == 3:
                    decisions, sft_prompt, sft_completion = result
                elif len(result) == 2:
                    decisions, sft_prompt = result
                    sft_completion = ""
                elif len(result) == 1:
                    decisions = result[0]
                    sft_prompt = ""
                    sft_completion = ""
                else:
                    decisions = result[0] if len(result) > 0 else []
                    sft_prompt = ""
                    sft_completion = ""
            elif isinstance(result, dict):
                decisions = result.get("decisions", [])
                sft_prompt = result.get("prompt", "")
                sft_completion = result.get("completion", "")
                tool_calls_made = result.get("tool_calls_made", 0)
            else:
                decisions = result
                sft_prompt = ""
                sft_completion = ""

            # Normalize decisions into list[dict]
            if isinstance(decisions, dict):
                decisions = [decisions]

            while isinstance(decisions, list) and len(decisions) == 1 and isinstance(decisions[0], list):
                decisions = decisions[0]

            if not isinstance(decisions, list):
                decisions = [decisions]

            decisions = [d for d in decisions if isinstance(d, dict)]

            if not decisions:
                print("Warning: no valid decisions returned by llm_agent_decide")
                continue

        except Exception as e:
            print(f"Error during decision generation: {e}")
            continue

        total_tool_calls += tool_calls_made

        print("\nModel Decisions:")
        loop_buys = 0
        loop_sells = 0
        loop_holds = 0

        for d in decisions:
            print(d)
            action = str(d.get("action", "hold")).lower()
            if action == "buy":
                loop_buys += 1
            elif action == "sell":
                loop_sells += 1
            else:
                loop_holds += 1

        total_buys += loop_buys
        total_sells += loop_sells
        total_holds += loop_holds

        print("\nDecision Summary:")
        print(f"Tool calls made: {tool_calls_made}")
        print(f"BUY: {loop_buys} | SELL: {loop_sells} | HOLD: {loop_holds}")

        # --------------------------------------------------------------
        # Step 6.5: Log to data collector
        # --------------------------------------------------------------
        if collector is not None:
            try:
                # FIXED: collector.log signature
                collector.log(
                    sft_prompt=sft_prompt,
                    sft_completion=sft_completion,
                    equity=state.get("equity", 0.0),
                    tool_calls=tool_calls_made,
                    decisions=decisions
                )

                # FIXED: store prompt for delayed attach_outcome()
                pending_outcomes.append({
                    "prompt": sft_prompt,
                    "loop": loop_count
                })

            except Exception as e:
                print(f"Warning: collector logging failed: {e}")

        # --------------------------------------------------------------
        # Step 7: Execute orders
        # --------------------------------------------------------------
        real_trades_this_loop = 0

        for d in decisions:
            action = str(d.get("action", "hold")).lower()

            if action in ["buy", "sell"]:
                try:
                    result = submit_order(tc, d)
                    print(f"Order submitted: {result}")
                    real_trades_this_loop += 1

                    if memory is not None:
                        situation = f"""
State equity: {state.get('equity', 0)}
Cash: {state.get('cash', 0)}
Positions: {state.get('positions', [])}

News:
{news_context}

Extra Context:
{extra_context}

Past reflections:
{past_reflections}

Decision:
{d}
""".strip()

                        pending_reflections.append({
                            "loop": loop_count,
                            "situation": situation,
                            "decision": d,
                            "equity": state["equity"]
                        })

                except Exception as e:
                    print(f"Order failed for {d.get('symbol', 'UNKNOWN')}: {e}")

            elif action == "hold":
                print(f"HOLD {d.get('symbol', 'UNKNOWN')}: {d.get('reason', 'No reason provided')}")

            else:
                print(f"Unknown action for decision: {d}")

        total_real_trades += real_trades_this_loop

        print("\nLoop Execution Summary:")
        print(f"Real trades executed: {real_trades_this_loop}")
        print(f"Pending reflections now: {len(pending_reflections)}")
        print(f"Pending collector outcomes now: {len(pending_outcomes)}")

    print("\n" + "=" * 60)
    print("TRADING LOOP COMPLETE")
    print("=" * 60)
    print(f"Total tool calls: {total_tool_calls}")
    print(f"Total real trades: {total_real_trades}")
    print(f"Total BUY decisions: {total_buys}")
    print(f"Total SELL decisions: {total_sells}")
    print(f"Total HOLD decisions: {total_holds}")

### Run the Trading Loop

Use the **fine-tuned model** (`lora_model`) from Task 4 for live paper trading.

In [55]:
# Initialize memory system
try:
    memory = TradingMemory("trading_memory.json")
    print(f"Memory loaded: {len(memory.memories)} past entries")
except:
    memory = None
    print("Memory not available (implement Task 3 first)")

# Initialize data collector
collector = TradingDataCollector()

# Use fine-tuned model from Task 4 (fall back to base model if not available)
try:
    trading_model = lora_model
    print("Using fine-tuned LoRA model from Task 4")
except NameError:
    trading_model = model
    print("Warning: lora_model not found, using base model (run Task 4 first)")

print("\nStarting trading loop...")
print(f"  Symbols: {len(WHITELIST)} stocks")
print(f"  Iterations: 80")
print(f"  Interval: {LOOP_SECONDS}s")
print("-" * 60)

try:
    run_trading_loop(tc, tok, trading_model, memory=memory,
                     collector=collector, n_iterations=80)
    print("\n\nTrading loop completed successfully!")
    print("\nData collected:")
    collector.summary()
except NotImplementedError:
    print("Task 5 not yet implemented")
except Exception as e:
    print(f"\nError: {e}")
    import traceback; traceback.print_exc()

Loaded 9 trading memories
Memory loaded: 9 past entries
Using fine-tuned LoRA model from Task 4

Starting trading loop...
  Symbols: 12 stocks
  Iterations: 80
  Interval: 10s
------------------------------------------------------------

LOOP #1 / 80 | Pending reflections: 0

Current Account State:
Equity: $99,208.12
Cash:   $110,128.27
Positions: 5
 - AAPL: qty=-6.0, value=$-1,549.80
 - AMZN: qty=3.0, value=$639.94
 - GOOG: qty=-2.0, value=$-595.62
 - GS: qty=-16.0, value=$-13,841.92
 - NVDA: qty=25.0, value=$4,427.25
  [Agent turn 2] -> get_news({'symbols': ['MSFT']})
  [Agent] Finalising — 3 turn(s), 1 tool call(s)
  [Agent] Finalising — 4 turn(s), 1 tool call(s)

Model Decisions:
{'action': 'buy', 'symbol': 'MSFT', 'qty': 1, 'order_type': 'market', 'limit_price': None, 'reason': 'Strong bullish RSI and MACD signals, combined with recent positive price movement.'}

Decision Summary:
Tool calls made: 1
BUY: 1 | SELL: 0 | HOLD: 0
Order failed for MSFT: {"code":40310000,"existing_order

### Optional: Inspect & Edit Memory

Use these cells to view, manually edit, download, or upload `trading_memory.json`.
You can add your own reflections or remove bad ones before the trading loop uses them.


In [56]:
# =============================================================================
# OPTIONAL: Inspect & Edit trading_memory.json
# Run each sub-block independently as needed.
# =============================================================================

import os, json
from google.colab import files

memory_path = os.path.join(CACHE_DIR, "trading_memory.json")

# -- 1. View all memory entries --------------------------------------------
if os.path.exists(memory_path):
    with open(memory_path) as f:
        memories = json.load(f)
    print(f"Found {len(memories)} memory entries:\n")
    for i, m in enumerate(memories):
        label = m.get('label', '?')
        pnl   = m.get('pnl', '?')
        refl  = m.get('reflection', '')[:120].replace('\n', ' ')
        print(f"[{i:2d}] label={label}  pnl={pnl}")
        print(f"      {refl}")
        print()
else:
    print("No trading_memory.json found yet. Run the backtest first.")

# -- 2. Download for local editing ----------------------------------------
# Uncomment to download:
# files.download(memory_path)

# -- 3. Upload an edited version back -------------------------------------
# Uncomment the block below after you have edited the file locally:
# uploaded = files.upload()   # select your edited trading_memory.json
# for fname, content in uploaded.items():
#     with open(memory_path, 'wb') as f:
#         f.write(content)
# with open(memory_path) as f:
#     memories = json.load(f)
# print(f"Reloaded {len(memories)} memories from uploaded file.")

# -- 4. Delete a bad entry by index ---------------------------------------
# Example: remove entry at index 2
# BAD_IDX = 2
# if os.path.exists(memory_path):
#     with open(memory_path) as f:
#         memories = json.load(f)
#     removed = memories.pop(BAD_IDX)
#     with open(memory_path, 'w') as f:
#         json.dump(memories, f, indent=2)
#     print(f"Removed entry {BAD_IDX}: {removed.get('reflection','')[:60]}")
#     print(f"{len(memories)} entries remain.")


No trading_memory.json found yet. Run the backtest first.


---
## Bonus Challenges (Optional)

### 1. DPO (Direct Preference Optimization)
Instead of SFT (training only on "good" examples), use **DPO** to train on preference pairs:

```python
from trl import DPOTrainer, DPOConfig

# Format: (prompt, chosen_response, rejected_response)
# "chosen" = decisions labeled "good"
# "rejected" = decisions labeled "bad"

dpo_dataset = Dataset.from_dict({
    "prompt": [e["prompt"] for e in pairs],
    "chosen": [e["good_completion"] for e in pairs],
    "rejected": [e["bad_completion"] for e in pairs],
})

config = DPOConfig(
    output_dir="dpo_output",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=5e-5,
)

trainer = DPOTrainer(
    model=lora_model,
    ref_model=model,       # frozen base model
    train_dataset=dpo_dataset,
    tokenizer=tok,
    args=config,
)
trainer.train()
```

**Key difference**: SFT teaches "do this", DPO teaches "prefer this over that" â€” generally produces better-calibrated models.

### 2. Vector Database RAG
Replace BM25 with semantic search:
- Use sentence embeddings (`sentence-transformers`)
- Store in FAISS vector index
- Retrieve by cosine similarity

### 3. Advanced Trading Strategy
- Position sizing based on confidence scores
- Stop-loss and take-profit logic
- Portfolio performance tracking with matplotlib

### 4. More Data Sources
- SEC filings (EDGAR API)
- Social media sentiment (Reddit, Twitter)
- Options chain data
- Earnings calendar integration